### Evaluate — Pipeline Comparison

Scores crop-based and/or YOLO pipeline outputs against CVAT ground-truth annotations.
Produces per-class precision / recall / F1 / AP, confusion matrices, PR curves, and an HTML report.

**Input** — `outputs/inference/crop_results/{RUN}/` and/or `outputs/inference/yolo_results/{RUN}/` · `data/evaluation/annotations/`  
**Output** — `outputs/evaluation/{run}_{ts}/summary.csv`, `confusion_matrix.png`, `pr_curve.png`, `threshold_analysis.png`, `report.html`

**Must edit (Cell 2):**

| Variable | Risk if skipped |
|----------|-----------------|
| `CROP_RUNS` / `YOLO_RUNS` | That pipeline skipped entirely if dict is empty or path wrong |
| `GT_CLASSES` / `CLASSES_NO_BB` | All detections appear as FPs if class order doesn't match inference |

**`CONF_THRESHOLDS`** (default `{'five_class_eff': 0, 'five_class_ins': 0, 'two_stage': 0.0}`) — applied **at eval time only**, CSV never touched. The five_class pipelines produce many FPs at threshold 0 because any non-background argmax counts as a detection. This pipeline prioritises **not missing insects (low FN)**, so choose the threshold from Cell 15's `threshold_analysis.png` based on how much precision loss you can accept. The table below shows example numbers from one run — your numbers will differ:

| threshold | FP removed | TP kept |
|-----------|------------|---------|
| 0 (off) | 0 % | 100 % |
| 0.30 | 71 % | 78 % |
| 0.40 | 96 % | 55 % |
| 0.50 | 98 % | 47 % |


##### Cell 1 — Environment  *(no edits needed)*

**Local:** auto-detects the repo root via `git rev-parse --show-toplevel` — no path editing required.

**Colab:** uses the path extracted from the zip in Cell 0.

Sets all derived paths (`MODEL_DIR`, `LABELED_DIR`, output folders).

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
from pathlib import Path
from datetime import datetime

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import zipfile, os
    DRIVE_ROOT  = Path('/content/drive/MyDrive')
    ZIP_PATH    = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO  = Path('/content/pollinator-colab')

    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} to /content/ ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('\u2713 Extracted')
    else:
        print('\u2713 Already extracted')

    BASE_DIR   = EXTRACT_TO
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    import subprocess as _sp
    _git_root  = Path(_sp.check_output(
        ['git', 'rev-parse', '--show-toplevel'], text=True).strip())
    BASE_DIR   = _git_root / 'ml_pipelines' / 'notebooks' / 'pollinator_detection'
    DRIVE_BASE = BASE_DIR

MODEL_DIR         = BASE_DIR  / 'models'
CROP_RESULTS_ROOT = (DRIVE_BASE if IN_COLAB else BASE_DIR) / 'outputs' / 'inference' / 'crop_results'
YOLO_RESULTS_ROOT = (DRIVE_BASE if IN_COLAB else BASE_DIR) / 'outputs' / 'inference' / 'yolo_results'

# ── Each run gets its own timestamped folder, old ones are never overwritten ─
RUN_TS   = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
EVAL_DIR = BASE_DIR / 'outputs' / 'evaluation' / RUN_TS
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env              : {"Colab" if IN_COLAB else "Local"}')
print(f'Run timestamp    : {RUN_TS}')
print(f'EVAL_DIR         : {EVAL_DIR}')
print(f'CROP_RESULTS_ROOT: {CROP_RESULTS_ROOT}  exists={CROP_RESULTS_ROOT.exists()}')
print(f'YOLO_RESULTS_ROOT: {YOLO_RESULTS_ROOT}  exists={YOLO_RESULTS_ROOT.exists()}')


##### Cell 2 — Config  ← **edit before every run**
Set `CROP_RUNS`, `YOLO_RUNS`, `GT_CLASSES`, and `CONF_THRESHOLDS`.

In [ ]:
# ── Ground truth data ────────────────────────────────────────────
# Default: evaluation set with CVAT annotations.
# Change if you want to evaluate against a different annotated set.
IMAGE_ROOT  = BASE_DIR / 'data' / 'evaluation' / 'images'
GT_ANN_ROOT = BASE_DIR / 'data' / 'evaluation' / 'annotations'

# ── Which runs to evaluate ──────────────────────────────────────
# Add one entry per inference run you want to compare.
# Key = short label used in plots; value = path to the run folder.
# Run names are auto-generated timestamps, e.g. 'run_20260522_143000'.
CROP_RUNS = {
    'final_crop_run':  CROP_RESULTS_ROOT / 'run_20260527_074717',
}
YOLO_RUNS = {
    'yolo': YOLO_RESULTS_ROOT / 'run_20260527_125448',
    # Add or remove entries freely — leave dict empty to skip YOLO evaluation
}

GT_CLASSES        = ['bumblebee', 'fly', 'butterfly', 'other']
CLASSES_NO_BB     = ['fly', 'butterfly', 'other']  # YOLO not trained on bumblebee
STRIP_HEIGHT = 120

# ── Confidence thresholds ──────────────────────────────────────────────
# Applied at evaluation time only — the CSV is never modified.
# Predictions below this confidence are treated as background for
# metric computation. Set to 0 to use all predictions as-is.
#
# Based on run_01 vs CVAT ground truth (five_class_ins):
#   thresh=0.40 → removes 96% of FPs, keeps 55% of TPs, precision 0.454
#   thresh=0.45 → removes 98% of FPs, keeps 52% of TPs, precision 0.595
#   thresh=0.50 → removes 98% of FPs, keeps 47% of TPs, precision 0.669
# See threshold_analysis.png (generated by Cell 10) for the full curve.
CONF_THRESHOLDS = {
    'five_class_eff': 0,   # ← adjust; 0 = off
    'five_class_ins': 0,   # ← adjust; 0 = off
    'two_stage':      0,   # two_stage uses binary stage as filter — no extra threshold needed
    'yolo':           0,   # ← adjust; 0 = off (YOLO threshold set at inference time via sahi_conf)
}

print('Runs to evaluate:')
for name, path in {**CROP_RUNS, **YOLO_RUNS}.items():
    print(f'  {name}: exists={path.exists()}')
print('\nConf thresholds (eval-time post-processing):')
for pipe, thr in CONF_THRESHOLDS.items():
    print(f'  {pipe}: {thr if thr else "off"}')

print('\n' + '='*60)
print('EVALUATION RUN SUMMARY')
print('='*60)
print(f'  Timestamp  : {RUN_TS}')
print('  Crop runs:')
for name, path in CROP_RUNS.items():
    print(f'    {name}: {path.name}  exists={path.exists()}')
print('  YOLO runs:')
for name, path in YOLO_RUNS.items():
    print(f'    {name}: {path.name}  exists={path.exists()}')
print('='*60)

# ── Save eval config to EVAL_DIR/eval_config.json ────────────────
import json as _json_cfg
_eval_config = {
    'eval_timestamp': RUN_TS,
    'crop_runs': {name: str(path) for name, path in CROP_RUNS.items()},
    'yolo_runs': {name: str(path) for name, path in YOLO_RUNS.items()},
    'conf_thresholds': CONF_THRESHOLDS,
    'gt_classes': GT_CLASSES,
    'strip_height': STRIP_HEIGHT,
}
(EVAL_DIR / 'eval_config.json').write_text(
    _json_cfg.dumps(_eval_config, indent=2)
)
print(f'  → eval_config.json saved to {EVAL_DIR}')


##### Cell 3 — Imports
Standard library imports. Just run.

In [ ]:
import csv, json as _json, io, sys
from pathlib import Path
from collections import defaultdict
import numpy as np, cv2
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Per-section stdout capture for report.json ──────────────────────
class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, data):
        for s in self.streams: s.write(data)
    def flush(self):
        for s in self.streams: s.flush()

_report_sections = {}   # section_name -> captured text
_current_buf     = None
_real_stdout     = sys.stdout

def _start_capture(section):
    global _current_buf
    _current_buf = io.StringIO()
    sys.stdout   = _Tee(_real_stdout, _current_buf)

def _stop_capture(section):
    global _current_buf
    sys.stdout = _real_stdout
    if _current_buf is not None:
        _report_sections[section] = _current_buf.getvalue()
        _current_buf = None

print('✓ Imports done')


##### Cell 4 — Load ground truth

Reads CVAT YOLO 1.1 annotations.
Uses **original image dimensions** for coordinate conversion (not stripped height).

In [ ]:
def load_gt(gt_ann_root, image_root, gt_classes, strip_height=120):
    print('Loading ground truth...')
    gt = {}
    for cam_dir in sorted(Path(gt_ann_root).iterdir()):
        if not cam_dir.is_dir(): continue
        lbl_dir = cam_dir / 'obj_train_data'
        img_dir = Path(image_root) / cam_dir.name
        if not lbl_dir.exists() or not img_dir.exists(): continue
        names_f = cam_dir / 'obj.names'
        cls_names = ([l.strip() for l in names_f.read_text().splitlines() if l.strip()]
                     if names_f.exists() else gt_classes)
        n_cam = 0
        for txt in sorted(lbl_dir.glob('*.txt')):
            img_p = None
            for ext in ('.JPG','.jpg','.jpeg','.png'):
                cand = img_dir / (txt.stem + ext)
                if cand.exists(): img_p = cand; break
            if img_p is None: continue
            img = cv2.imread(str(img_p))
            if img is None: continue
            H_orig, W = img.shape[:2]  # always use ORIGINAL dims
            boxes = []
            for line in txt.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5: continue
                try: ci,cx,cy,bw,bh = int(parts[0]),*[float(x) for x in parts[1:5]]
                except ValueError: continue
                cname = cls_names[ci] if ci < len(cls_names) else str(ci)
                y1 = (cy - bh/2) * H_orig
                y2 = (cy + bh/2) * H_orig
                # skip boxes entirely in OSD strip
                if strip_height > 0 and y1 >= (H_orig - strip_height): continue
                y2 = min(y2, H_orig - strip_height)
                boxes.append({'cls': cname,
                              'x1': (cx-bw/2)*W, 'y1': y1,
                              'x2': (cx+bw/2)*W, 'y2': y2})
            gt[str(img_p)] = boxes
            n_cam += len(boxes)
        if n_cam: print(f'  {cam_dir.name}: {n_cam} annotations')
    n_total = sum(len(v) for v in gt.values())
    cls_cnt = {}
    for boxes in gt.values():
        for b in boxes: cls_cnt[b['cls']] = cls_cnt.get(b['cls'],0)+1
    print(f'\n✓ GT: {len(gt)} images  {n_total} annotations')
    for c,n in sorted(cls_cnt.items()): print(f'  {c:15}: {n}')
    return gt

gt = load_gt(GT_ANN_ROOT, IMAGE_ROOT, GT_CLASSES, STRIP_HEIGHT)
gt_with_boxes = {k:v for k,v in gt.items() if len(v) > 0}
print(f'Images with annotations: {len(gt_with_boxes)}')


##### Cell 5 — Load inference results

Loads all crop and YOLO results.
**Crop results**: every candidate bbox row (both insect AND background rejected).
**YOLO results**: every detection from yolo_results.csv.

In [ ]:
def detect_pipelines(csv_path):
    with open(csv_path, newline='') as f:
        fields = csv.DictReader(f).fieldnames or []
    return sorted({f.split('__')[0] for f in fields
                   if '__binary_label' in f or '__pollinator_type' in f})

def load_crop_run(run_path, image_root):
    run_path = Path(run_path)
    cfg_file = run_path / 'run_config.json'
    run_cfg  = _json.loads(cfg_file.read_text()) if cfg_file.exists() else {}
    rows = []; pipe_names = []
    for csv_path in sorted(run_path.rglob('results.csv')):
        cam_name = csv_path.parent.name
        img_dir  = Path(image_root) / cam_name
        if not pipe_names:
            pipe_names = detect_pipelines(csv_path)
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                img_name = row.get('image_name', '')
                # Build full path from camera_folder + image_name
                img_p = str(img_dir / img_name)
                if not Path(img_p).exists():
                    stem = Path(img_name).stem
                    for ext in ('.JPG','.jpg','.jpeg'):
                        cand = str(img_dir / (stem+ext))
                        if Path(cand).exists(): img_p=cand; break
                row['_img_path'] = img_p
                row['_cam']      = cam_name
                rows.append(row)
    pre = run_cfg.get('preprocess', {})
    print(f'  Pipelines : {pipe_names}')
    print(f'  Rows      : {len(rows)}')
    print(f'  Config    : large_motion={pre.get("enable_large_motion","?")}  '
          f'darker_threshold={pre.get("darker_threshold","?")}')
    return rows, pipe_names, run_cfg

def load_yolo_run(run_path, image_root):
    run_path = Path(run_path)
    cfg_file = run_path / 'run_config.json'
    run_cfg  = _json.loads(cfg_file.read_text()) if cfg_file.exists() else {}
    rows = []
    for csv_path in sorted(run_path.rglob('yolo_results.csv')):
        cam_name = csv_path.parent.name
        img_dir  = Path(image_root) / cam_name
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                img_name = row.get('image_name', '')
                img_p = str(img_dir / img_name)
                if not Path(img_p).exists():
                    stem = Path(img_name).stem
                    for ext in ('.JPG','.jpg','.jpeg'):
                        cand = str(img_dir / (stem+ext))
                        if Path(cand).exists(): img_p=cand; break
                row['_img_path'] = img_p
                row['_cam']      = cam_name
                rows.append(row)
    print(f'  YOLO detections: {len(rows)}')
    return rows, run_cfg

print('Loading crop runs...')
crop_run_data = {}
for name, path in CROP_RUNS.items():
    print(f'\n  {name}:')
    rows, pipes, cfg = load_crop_run(path, IMAGE_ROOT)
    crop_run_data[name] = {'rows':rows,'pipes':pipes,'config':cfg}

print('\nLoading YOLO runs...')
yolo_run_data = {}
for name, path in YOLO_RUNS.items():
    print(f'\n  {name}:')
    rows, cfg = load_yolo_run(path, IMAGE_ROOT)
    yolo_run_data[name] = {'rows':rows,'config':cfg}

print('\n✓ All runs loaded.')


##### Cell 6 — Matching + metrics functions

**Do not edit.** Defines all evaluation logic.

**`center_match`**: matches predictions to GT using three criteria (any one sufficient):
1. GT center falls inside pred bbox
2. Pred center falls inside GT bbox
3. Overlap area / GT area ≥ 20%

**`evaluate_one_pipeline`**: for one pipeline computes:
- **TP** — bbox matched GT (any criterion above) + correct class
- **FP** — bbox with no matching GT (false alarm)
- **FN** — GT bbox with no matching prediction (missed insect)
- **FN breakdown**:
  - `fn_detected_as_bg` — pipeline HAD a bbox near this insect but classified it as background
  - `fn_not_detected` — no bbox at all near this insect (frame-diff completely missed it)
- **bg_rejected** — total candidate crops classified as background


In [ ]:
def bbox_overlap_ratio(p, g):
    ix1=max(p['x1'],g['x1']); iy1=max(p['y1'],g['y1'])
    ix2=min(p['x2'],g['x2']); iy2=min(p['y2'],g['y2'])
    iw=max(0,ix2-ix1); ih=max(0,iy2-iy1)
    gt_area=max(1,(g['x2']-g['x1'])*(g['y2']-g['y1']))
    return iw*ih/gt_area

def center_match(pred_boxes, gt_boxes, overlap_thresh=0.20):
    """Three criteria: GT center in pred, pred center in GT, or 20% overlap."""
    def inside(px,py,x1,y1,x2,y2): return x1<=px<=x2 and y1<=py<=y2
    mp=set(); mg=set(); pairs=[]
    for gi,g in enumerate(gt_boxes):
        gcx=(g['x1']+g['x2'])/2; gcy=(g['y1']+g['y2'])/2
        for pi,p in enumerate(pred_boxes):
            if pi in mp: continue
            pcx=(p['x1']+p['x2'])/2; pcy=(p['y1']+p['y2'])/2
            if (inside(gcx,gcy,p['x1'],p['y1'],p['x2'],p['y2']) or
                inside(pcx,pcy,g['x1'],g['y1'],g['x2'],g['y2']) or
                bbox_overlap_ratio(p,g)>=overlap_thresh):
                pairs.append((pi,gi)); mp.add(pi); mg.add(gi); break
    return pairs,[i for i in range(len(pred_boxes)) if i not in mp],\
                 [i for i in range(len(gt_boxes)) if i not in mg]

def evaluate_one_pipeline(preds_by_img, gt, classes, label):
    """
    Two-level evaluation:
    1. Detection: did pipeline find a bbox near the insect? (class-agnostic)
    2. Classification: was the class correct? (among detected)

    Also tracks:
    - fn_detected_as_bg: GT insects that WERE detected but classified as background
    - fn_not_detected:   GT insects that had NO bbox near them at all
    """
    rows_out=[]; n_bg_rejected=0
    det_tp=det_fp=det_fn=0
    cls_correct=0; cls_wrong=0
    tp_c=defaultdict(int); fp_c=defaultdict(int); fn_c=defaultdict(int)
    cls_confusion=defaultdict(lambda: defaultdict(int))

    # FN breakdown
    fn_detected_as_bg=0   # had a bbox but was rejected as background
    fn_not_detected=0     # no bbox at all near this GT insect

    for img_p, gt_boxes in gt.items():
        all_preds = preds_by_img.get(img_p, [])
        insect    = [p for p in all_preds if not p.get('is_bg')]
        rejected  = [p for p in all_preds if p.get('is_bg')]
        n_bg_rejected += len(rejected)

        # Match insect predictions to GT
        pairs, unp, ung = center_match(insect, gt_boxes)

        for pi,gi in pairs:
            det_tp+=1
            pc=insect[pi]['cls']; gc=gt_boxes[gi]['cls']
            correct=(pc==gc)
            if correct: cls_correct+=1; tp_c[gc]+=1
            else: cls_wrong+=1; fp_c[pc]+=1; fn_c[gc]+=1
            cls_confusion[gc][pc]+=1
            rows_out.append({'pipeline':label,'img':img_p,'match':'tp',
                             'pred_cls':pc,'gt_cls':gc,
                             'conf':insect[pi]['conf'],'correct_cls':correct})
        for pi in unp:
            det_fp+=1; fp_c[insect[pi]['cls']]+=1
            rows_out.append({'pipeline':label,'img':img_p,'match':'fp',
                             'pred_cls':insect[pi]['cls'],'gt_cls':'',
                             'conf':insect[pi]['conf'],'correct_cls':False})

        # For each unmatched GT, check if a REJECTED bbox covers it
        for gi in ung:
            det_fn+=1; fn_c[gt_boxes[gi]['cls']]+=1
            g = gt_boxes[gi]
            # Check if any bg_rejected bbox overlaps this GT
            covered_by_bg = any(
                bbox_overlap_ratio(r, g) >= 0.20 or
                bbox_overlap_ratio(g, r) >= 0.20
                for r in rejected
            )
            if covered_by_bg:
                fn_detected_as_bg+=1
                match_type='fn_detected_as_bg'
            else:
                fn_not_detected+=1
                match_type='fn_not_detected'
            rows_out.append({'pipeline':label,'img':img_p,'match':match_type,
                             'pred_cls':'bg','gt_cls':gt_boxes[gi]['cls'],
                             'conf':0.0,'correct_cls':False})

    det_prec=det_tp/max(1,det_tp+det_fp)
    det_rec =det_tp/max(1,det_tp+det_fn)
    det_f1  =2*det_prec*det_rec/max(1e-8,det_prec+det_rec)
    cls_acc =cls_correct/max(1,det_tp)

    print(f'  [Detection]      P={det_prec:.3f}  R={det_rec:.3f}  F1={det_f1:.3f}  '
          f'TP={det_tp}  FP={det_fp}  FN={det_fn}  bg_rejected={n_bg_rejected}')
    print(f'  [FN breakdown]   detected_as_bg={fn_detected_as_bg}  '
          f'not_detected={fn_not_detected}  '
          f'({100*fn_detected_as_bg/max(1,det_fn):.1f}% were detected but rejected)')
    print(f'  [Classification] accuracy={cls_acc:.3f}  '
          f'correct={cls_correct}  wrong={cls_wrong}  (of {det_tp} detected)')
    print(f'  Per-class:')
    for c in classes:
        if tp_c[c] or fp_c.get(c) or fn_c[c]:
            print(f'    {c:15} TP={tp_c[c]:>4}  FP={fp_c.get(c,0):>4}  FN={fn_c[c]:>4}')

    return {'det_precision':det_prec,'det_recall':det_rec,'det_f1':det_f1,
            'det_tp':det_tp,'det_fp':det_fp,'det_fn':det_fn,
            'fn_detected_as_bg':fn_detected_as_bg,
            'fn_not_detected':fn_not_detected,
            'cls_accuracy':cls_acc,'cls_correct':cls_correct,'cls_wrong':cls_wrong,
            'n_bg_rejected':n_bg_rejected,
            'tp_c':dict(tp_c),'fp_c':dict(fp_c),'fn_c':dict(fn_c),
            'cls_confusion':dict(cls_confusion),
            'rows':rows_out}


##### Cell 7 — Build prediction index

Organises all predictions by image path for efficient matching.

For crop pipelines: every candidate bbox is included, both insect and background predictions.
`is_bg=True` means the pipeline classified this bbox as background.

For YOLO: every detection row from `yolo_results.csv`.


In [ ]:
def build_crop_index(rows, pipe_name, conf_threshold=0.0):
    """Build {img_path -> list of pred dicts} for one pipeline.

    conf_threshold (float): predictions below this confidence are treated
        as background at eval time. The CSV is not modified.
        Set from CONF_THRESHOLDS in Cell 2; 0 = use all predictions.
    """
    idx = defaultdict(list)
    for r in rows:
        if r.get('pollinator_detected') not in ('yes',): continue
        try:
            x=int(r['bbox_x']); y=int(r['bbox_y'])
            w=int(r['bbox_w']); h=int(r['bbox_h'])
        except (ValueError,KeyError): continue
        p = pipe_name + '__'
        bl  = r.get(p+'binary_label', '')
        pt  = r.get(p+'pollinator_type', '')
        # Determine if this bbox is insect or background for this pipeline
        is_bg = (bl == 'background') or (pt == 'background') or \
                (not bl and not pt)
        pred_cls = pt if (pt and pt != 'background') else (bl if bl == 'insect' else 'background')
        try: conf = float(r.get(p+'group_conf') or r.get(p+'binary_conf') or 0)
        except: conf = 0.0
        # Apply eval-time confidence threshold
        if not is_bg and conf_threshold > 0 and conf < conf_threshold:
            is_bg    = True
            pred_cls = 'background'
        idx[r['_img_path']].append({
            'x1':x,'y1':y,'x2':x+w,'y2':y+h,
            'cls':pred_cls,'conf':conf,'is_bg':is_bg
        })
    return idx

def build_yolo_index(rows, conf_threshold=0.0):
    idx = defaultdict(list)
    for r in rows:
        try:
            x=int(r['bbox_x']); y=int(r['bbox_y'])
            w=int(r['bbox_w']); h=int(r['bbox_h'])
            conf=float(r.get('confidence',0))
        except (ValueError,KeyError): continue
        if conf_threshold > 0 and conf < conf_threshold: continue
        idx[r['_img_path']].append({
            'x1':x,'y1':y,'x2':x+w,'y2':y+h,
            'cls':r.get('class_name',''),'conf':conf,'is_bg':False
        })
    return idx

# Build all indexes
pred_indexes = {}  # label -> {img_path -> [preds]}

for run_name, run_data in crop_run_data.items():
    for pipe_name in run_data['pipes']:
        thr   = CONF_THRESHOLDS.get(pipe_name, 0.0)
        label = f'{run_name}/{pipe_name}'
        pred_indexes[label] = build_crop_index(run_data['rows'], pipe_name,
                                               conf_threshold=thr)
        n = sum(len(v) for v in pred_indexes[label].values())
        n_ins = sum(1 for v in pred_indexes[label].values() for p in v if not p['is_bg'])
        n_bg  = sum(1 for v in pred_indexes[label].values() for p in v if p['is_bg'])
        print(f'  {label}: {n} total  insect={n_ins}  background={n_bg}')

for run_name, run_data in yolo_run_data.items():
    label = run_name
    thr = CONF_THRESHOLDS.get('yolo', 0.0)
    pred_indexes[label] = build_yolo_index(run_data['rows'], conf_threshold=thr)
    n = sum(len(v) for v in pred_indexes[label].values())
    print(f'  {label}: {n} detections')

print(f'\n✓ Prediction indexes built for {len(pred_indexes)} pipelines.')


##### Cell 8 — Run evaluation  ← main evaluation cell

Evaluates every pipeline against GT. Prints results as it goes.


In [ ]:
all_results = {}

for label, preds_by_img in pred_indexes.items():
    print(f'\n=== {label} ===')
    all_results[label] = evaluate_one_pipeline(
        preds_by_img, gt, GT_CLASSES, label)

print(f'\n✓ Evaluated {len(all_results)} pipelines.')

# ── Additional: all pipelines evaluated excluding bumblebee ──────
# (fair comparison since YOLO was not trained on bumblebee)
print('\n' + '='*70)
print('All pipelines — bumblebee EXCLUDED from GT (fair YOLO comparison)')
print('='*70)

# CLASSES_NO_BB defined in Cell 4
gt_no_bb = {img_p: [b for b in boxes if b['cls'] in CLASSES_NO_BB]
            for img_p, boxes in gt.items()}

for label, preds_by_img in pred_indexes.items():
    print(f'\n=== {label} ===')
    all_results[label + '_no_bb'] = evaluate_one_pipeline(
        preds_by_img, gt_no_bb, CLASSES_NO_BB, label + '_no_bb')


##### Cell 9 — Results table + confusion matrix + save

Prints:
1. **Overall detection metrics** (class-agnostic P/R/F1) with FN breakdown
2. **Per-class P/R/F1**
3. **Classification accuracy** among detected insects
4. **Confusion matrix** (GT class vs predicted class, including bg_rejected and missed)

Saves `eval_results.csv` and `summary.json` to `outputs/evaluation/`.


In [ ]:
_start_capture('overall_metrics')
import csv as _csv
import json as _json
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

# Split results into two groups for cleaner display
_regular = {k: v for k, v in all_results.items() if not k.endswith('_no_bb')}
_no_bb   = {k: v for k, v in all_results.items() if k.endswith('_no_bb')}

def _print_overall_table(results):
    print(f'{"Pipeline":40}  {"Precision":>10}  {"Recall":>7}  {"F1_Score":>9}  {"True_Pos":>9}  {"False_Pos":>10}  {"False_Neg":>10}  {"BG_Rejected":>12}')
    print('-'*105)
    for label, r in results.items():
        print(f'{label:40}  {r["det_precision"]:>10.3f}  {r["det_recall"]:>7.3f}  '
              f'{r["det_f1"]:>9.3f}  {r["det_tp"]:>9}  {r["det_fp"]:>10}  '
              f'{r["det_fn"]:>10}  {r["n_bg_rejected"]:>12}')

def _print_perclass_table(results, classes):
    for label, r in results.items():
        print(f'\n  {label}')
        print(f'  {"Class":15}  {"Precision":>10}  {"Recall":>7}  {"F1_Score":>9}  {"True_Pos":>9}  {"False_Pos":>10}  {"False_Neg":>10}')
        print(f'  {"-"*75}')
        for c in classes:
            tp = r["tp_c"].get(c, 0)
            fp = r["fp_c"].get(c, 0)
            fn = r["fn_c"].get(c, 0)
            p  = tp / max(1, tp+fp)
            rc = tp / max(1, tp+fn)
            f1 = 2*p*rc / max(1e-8, p+rc)
            print(f'  {c:15}  {p:>10.3f}  {rc:>7.3f}  {f1:>9.3f}  {tp:>9}  {fp:>10}  {fn:>10}')

def _print_cls_accuracy(results):
    print(f'{"Pipeline":40}  {"Accuracy":>9}  {"Correct":>8}  {"Wrong":>7}')
    print('-'*70)
    for label, r in results.items():
        print(f'{label:40}  {r["cls_accuracy"]:>9.3f}  {r["cls_correct"]:>8}  {r["cls_wrong"]:>7}')

def _print_confusion(results, classes):
    col_w = 20
    cols  = classes + ['background_rejected', 'missed']
    _hdr  = "Ground Truth / Prediction"
    header = f'  {_hdr:26}' + ''.join(f'{c:>{col_w}}' for c in cols)
    for label, r in results.items():
        print(f'\n  {label}')
        matrix = defaultdict(lambda: defaultdict(int))
        missed = defaultdict(int)
        bg_rej = defaultdict(int)
        for row in r['rows']:
            if row['match'] == 'tp':
                matrix[row['gt_cls']][row['pred_cls']] += 1
            elif row['match'] == 'fn':
                missed[row['gt_cls']] += 1
        for c in classes:
            bg_rej[c] = r['fn_c'].get(c, 0) - missed[c]
            if bg_rej[c] < 0: bg_rej[c] = 0
        print(header)
        print('  ' + '-'*(26 + col_w*len(cols)))
        for gt_c in classes:
            row_str = f'  {gt_c:26}'
            for pred_c in classes:
                row_str += f'{matrix[gt_c][pred_c]:>{col_w}}'
            row_str += f'{bg_rej[gt_c]:>{col_w}}'
            row_str += f'{missed[gt_c]:>{col_w}}'
            print(row_str)

# ── Section 1: all 4 classes (includes bumblebee) ────────────────
print('\n' + '█'*105)
print('  ALL CLASSES  (bumblebee · fly · butterfly · other)')
print('█'*105)

print('\n' + '='*105)
print('Overall Detection Metrics')
print('='*105)
_print_overall_table(_regular)

print('\n' + '='*105)
print('Per-class Detection Metrics')
print('='*105)
_print_perclass_table(_regular, GT_CLASSES)

print('\n' + '='*80)
print('Classification Accuracy (among detected insects)')
print('='*80)
_print_cls_accuracy(_regular)

print('\n' + '='*80)
print('Confusion Matrices')
print('='*80)
_print_confusion(_regular, GT_CLASSES)

# ── Section 2: bumblebee excluded (fair YOLO comparison) ─────────
print('\n\n' + '█'*105)
print('  BUMBLEBEE EXCLUDED  (fly · butterfly · other) — fair comparison since YOLO was not trained on bumblebee')
print('█'*105)

print('\n' + '='*105)
print('Overall Detection Metrics')
print('='*105)
_print_overall_table(_no_bb)

print('\n' + '='*105)
print('Per-class Detection Metrics')
print('='*105)
_print_perclass_table(_no_bb, CLASSES_NO_BB)

print('\n' + '='*80)
print('Classification Accuracy (among detected insects)')
print('='*80)
_print_cls_accuracy(_no_bb)

print('\n' + '='*80)
print('Confusion Matrices')
print('='*80)
_print_confusion(_no_bb, CLASSES_NO_BB)

# ── Build xlsx data ───────────────────────────────────────────────
_xls_overall = []
for label, r in all_results.items():
    _xls_overall.append({
        'pipeline':               label,
        'bumblebee_excluded':     label.endswith('_no_bb'),
        'precision':              round(r['det_precision'], 4),
        'recall':                 round(r['det_recall'],    4),
        'f1_score':               round(r['det_f1'],        4),
        'true_positives':         r['det_tp'],
        'false_positives':        r['det_fp'],
        'false_negatives':        r['det_fn'],
        'fn_detected_as_background': r['fn_detected_as_bg'],
        'fn_not_detected':        r['fn_not_detected'],
        'classification_accuracy': round(r['cls_accuracy'], 4),
        'correct':                r['cls_correct'],
        'wrong':                  r['cls_wrong'],
        'background_rejected':    r['n_bg_rejected'],
    })

_xls_per_class = []
for label, r in all_results.items():
    classes = CLASSES_NO_BB if label.endswith('_no_bb') else GT_CLASSES
    for c in classes:
        tp = r['tp_c'].get(c, 0); fp = r['fp_c'].get(c, 0); fn = r['fn_c'].get(c, 0)
        p = tp/max(1,tp+fp); rc = tp/max(1,tp+fn); f1 = 2*p*rc/max(1e-8,p+rc)
        _xls_per_class.append({
            'pipeline':            label,
            'bumblebee_excluded':  label.endswith('_no_bb'),
            'class':               c,
            'precision':           round(p,  4),
            'recall':              round(rc, 4),
            'f1_score':            round(f1, 4),
            'true_positives':      tp,
            'false_positives':     fp,
            'false_negatives':     fn,
        })

_xls_confusion = []
_cm_cnt = Counter()
for r in all_results.values():
    for row in r['rows']:
        if row['match'] == 'tp':
            _cm_cnt[(row['pipeline'], row['gt_cls'], row['pred_cls'])] += 1
        elif row['match'] in ('fn_detected_as_bg', 'fn_not_detected'):
            _col = 'background_rejected' if row['match'] == 'fn_detected_as_bg' else 'missed'
            _cm_cnt[(row['pipeline'], row['gt_cls'], _col)] += 1
for (pipe, gt_c, pred_c), cnt in sorted(_cm_cnt.items()):
    _xls_confusion.append({
        'pipeline':           pipe,
        'ground_truth_class': gt_c,
        'predicted_class':    pred_c,
        'count':              cnt,
    })

_xls_raw = []
for r in all_results.values():
    for row in r['rows']:
        _xls_raw.append({
            'pipeline':           row['pipeline'],
            'image':              row['img'],
            'match_type':         row['match'],
            'predicted_class':    row['pred_cls'],
            'ground_truth_class': row['gt_cls'],
            'confidence':         row['conf'],
            'correct_class':      row['correct_cls'],
        })

print(f'\n\u2713 Metrics collected (will be saved to xlsx at end of Cell 14)')
_stop_capture('overall_metrics')


##### Cell 10 — Confidence Threshold Analysis

For each pipeline, shows how P/R/F1 change at different confidence thresholds.

**Goal:** find the threshold that achieves target recall (≥0.8) with best precision.
Since the goal is not to miss insects, we prioritise recall.

Saves `threshold_analysis.png` to `outputs/evaluation/`.


In [ ]:
# ── Threshold analysis ────────────────────────────────────────────
RECALL_TARGETS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

print('\n' + '='*75)
print('Threshold Analysis — Recall-constrained')
print('(find best threshold for each recall target)')
print('='*75)

plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12,
    'axes.labelsize': 11, 'legend.fontsize': 9, 'xtick.labelsize': 10,
    'ytick.labelsize': 10})
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = plt.cm.tab10.colors

for ci, (label, r) in enumerate(all_results.items()):
    color = colors[ci % len(colors)]

    # Get all (conf, is_tp) pairs from rows
    dets = sorted(
        [(row['conf'], row['match'] == 'tp')
         for row in r['rows'] if row['match'] in ('tp', 'fp')],
        key=lambda x: -x[0])
    if not dets: continue

    n_gt = r['det_tp'] + r['det_fn']
    thresholds=[]; precisions=[]; recalls=[]; f1s=[]
    tp_ = fp_ = 0

    for conf, is_tp in dets:
        if is_tp: tp_ += 1
        else: fp_ += 1
        p  = tp_ / max(1, tp_+fp_)
        rc = tp_ / max(1, n_gt)
        f1 = 2*p*rc / max(1e-8, p+rc)
        thresholds.append(conf)
        precisions.append(p)
        recalls.append(rc)
        f1s.append(f1)

    # PR curve
    axes[0].plot(recalls, precisions, color=color,
                 label=f'{label} (F1={r["det_f1"]:.3f})', lw=2.0)

    # Threshold vs Recall/Precision
    axes[1].plot(thresholds, recalls, color=color, ls='-', lw=2,
                 label=f'{label} recall')
    axes[1].plot(thresholds, precisions, color=color, ls='--', lw=1,
                 alpha=0.6)

    # Find best threshold for each recall target
    print(f'\n  {label}')
    print(f'  {"Recall target":15}  {"Threshold":>10}  {"Precision":>10}  {"F1":>8}')
    print(f'  {"-"*50}')
    for target in RECALL_TARGETS:
        # Find highest threshold that achieves this recall
        best_thr = best_p = best_f1 = None
        for thr, p, rc, f1 in zip(thresholds, precisions, recalls, f1s):
            if rc >= target:
                if best_thr is None or thr > best_thr:
                    best_thr = thr; best_p = p; best_f1 = f1
        if best_thr is not None:
            print(f'  R≥{target:.2f}          {best_thr:>10.3f}  {best_p:>10.3f}  {best_f1:>8.3f}')
        else:
            print(f'  R≥{target:.2f}          {"N/A":>10}  {"N/A":>10}  {"N/A":>8}')

    # Mark optimal F1 point
    best_idx = int(np.argmax(f1s))
    axes[0].plot(recalls[best_idx], precisions[best_idx],
                 'o', color=color, ms=8)
    axes[0].annotate(f'  thr={thresholds[best_idx]:.2f}',
                     (recalls[best_idx], precisions[best_idx]),
                     fontsize=8, color=color)

axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('PR Curve (● = best F1)')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0,1); axes[0].set_ylim(0,1)

axes[1].set_xlabel('Confidence Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Recall (—) and Precision (--) vs Threshold')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)

# Mark recall=0.8 line
axes[0].axhline(y=0, color='grey', ls=':', alpha=0.5)
axes[1].axhline(y=0.8, color='grey', ls=':', lw=1, label='R=0.8 target')

plt.suptitle('Confidence Threshold Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(EVAL_DIR/'threshold_analysis.png', dpi=300, bbox_inches='tight')
print(f'\n✓ Saved threshold_analysis.png')
plt.show()

# ── Collect threshold data for xlsx ───────────────────────────────
_xls_threshold = []
for _label, _r in all_results.items():
    _dets = sorted(
        [(_row['conf'], _row['match'] == 'tp')
         for _row in _r['rows'] if _row['match'] in ('tp', 'fp')],
        key=lambda x: -x[0])
    if not _dets: continue
    _n_gt = _r['det_tp'] + _r['det_fn']
    _tp2 = _fp2 = 0
    for _conf, _is_tp in _dets:
        if _is_tp: _tp2 += 1
        else: _fp2 += 1
        _p2  = _tp2 / max(1, _tp2 + _fp2)
        _rc2 = _tp2 / max(1, _n_gt)
        _f12 = 2 * _p2 * _rc2 / max(1e-8, _p2 + _rc2)
        _xls_threshold.append({'pipeline':_label,
            'threshold':round(_conf,4),'precision':round(_p2,4),
            'recall':round(_rc2,4),'f1':round(_f12,4),'tp':_tp2,'fp':_fp2})


##### Cell 11 — Browse images with all pipeline bboxes

Displays original images with GT + all pipeline bboxes drawn in memory.
**No files are saved.** Uses matplotlib for cross-platform keyboard navigation.

**Keys:** ← → to navigate, q to quit.

Run `%matplotlib tk` in a separate cell first if using locally.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2, numpy as np
from pathlib import Path
from collections import defaultdict

# ── Build display index: img_path -> list of (label,x1,y1,x2,y2,color,thick) ──
display_index = defaultdict(list)

COLORS = {
    'GT':             (0,   0.78, 0),      # green
    'two_stage':      (1,   0.39, 0),      # blue-ish orange
    'five_class_eff': (0,   0.55, 1),      # orange
    'five_class_ins': (0.7, 0,   1),       # purple
    'yolo_run_01':    (1,   0,   0),       # red
}
DEFAULT_COLOR = (0.5, 0.5, 0.5)

# GT
for img_p, boxes in gt_with_boxes.items():
    for b in boxes:
        display_index[img_p].append(
            (f'GT:{b["cls"]}', b['x1'],b['y1'],b['x2'],b['y2'],
             COLORS['GT'], 2))

# All pipelines
for label, preds_by_img in pred_indexes.items():
    pipe_key = label.split('/')[-1] if '/' in label else label
    color = COLORS.get(pipe_key, DEFAULT_COLOR)
    for img_p, preds in preds_by_img.items():
        for p in preds:
            is_bg = p.get('is_bg', False)
            thick = 0.5 if is_bg else 2
            lbl   = '' if is_bg else f'{pipe_key}:{p["cls"]}'
            c     = (0.7,0.7,0.7) if is_bg else color
            display_index[img_p].append(
                (lbl, p['x1'],p['y1'],p['x2'],p['y2'], c, thick))

img_list = [p for p in sorted(display_index.keys()) if display_index[p]]
print(f'{len(img_list)} images with detections or GT')
print('Keys: ← → to navigate, q to quit')

idx = [0]
fig, ax = plt.subplots(figsize=(16, 9))
plt.subplots_adjust(top=0.95, bottom=0.02)

def draw(i):
    ax.clear()
    img_p = img_list[i]
    img   = cv2.imread(img_p)
    if img is None: return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    for lbl,x1,y1,x2,y2,color,thick in display_index[img_p]:
        rect = mpatches.Rectangle(
            (x1,y1), x2-x1, y2-y1,
            linewidth=thick, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        if lbl:
            ax.text(x1, max(y1-4,10), lbl,
                    color=color, fontsize=6,
                    bbox=dict(fc='black', alpha=0.4, pad=1, ec='none'))
    name = f'{Path(img_p).parent.name}/{Path(img_p).name}'
    ax.set_title(f'[{i+1}/{len(img_list)}]  {name}', fontsize=9)
    ax.axis('off')
    fig.canvas.draw_idle()

def on_key(event):
    if event.key == 'right':
        idx[0] = min(len(img_list)-1, idx[0]+1); draw(idx[0])
    elif event.key == 'left':
        idx[0] = max(0, idx[0]-1); draw(idx[0])
    elif event.key == 'q':
        plt.close()

fig.canvas.mpl_connect('key_press_event', on_key)
draw(0)
plt.show()


In [ ]:
plt.show()


##### Cell 12 — Per-plot (per camera folder) metrics

Shows detection metrics broken down by each camera folder.
Useful for spotting which plots are harder for each pipeline.


In [ ]:
from collections import defaultdict as _dd12

print('\n' + '='*100)
print('Per-plot Detection Metrics')
print('='*100)

_xls_plot_det = []
_xls_plot_cls = []

_start_capture('per_plot_metrics')

def _perplot_section(pred_indexes_subset, section_classes, gt_dict=None, bumblebee_excluded=False):
    if gt_dict is None:
        gt_dict = gt
    for label, preds_by_img in pred_indexes_subset.items():
        print(f'\n{"="*70}')
        print(f'  {label}')
        print(f'{"="*70}')

        plots = {}
        for img_p, gt_boxes in gt_dict.items():
            cam = Path(img_p).parent.name
            plots.setdefault(cam, {'gt': {}, 'preds': {}})
            plots[cam]['gt'][img_p] = gt_boxes
        for img_p, preds in preds_by_img.items():
            cam = Path(img_p).parent.name
            if cam in plots:
                plots[cam]['preds'][img_p] = preds

        print(f'\n  [Detection — class agnostic]')
        print(f'  {"Plot":45}  {"Precision":>10}  {"Recall":>7}  {"F1_Score":>9}  {"True_Pos":>9}  {"False_Pos":>10}  {"False_Neg":>10}  {"Ground_Truth":>13}')
        print(f'  {"-"*115}')
        for cam in sorted(plots.keys()):
            cam_gt    = plots[cam]['gt']
            cam_preds = plots[cam]['preds']
            tp=fp=fn=0
            for img_p, gt_boxes in cam_gt.items():
                insect = [p for p in cam_preds.get(img_p,[]) if not p.get('is_bg')]
                pairs, unp, ung = center_match(insect, gt_boxes)
                tp+=len(pairs); fp+=len(unp); fn+=len(ung)
            p=tp/max(1,tp+fp); r=tp/max(1,tp+fn); f1=2*p*r/max(1e-8,p+r)
            print(f'  {cam:45}  {p:>10.3f}  {r:>7.3f}  {f1:>9.3f}  {tp:>9}  {fp:>10}  {fn:>10}  {tp+fn:>13}')
            _xls_plot_det.append({
                'pipeline':             label,
                'bumblebee_excluded':   bumblebee_excluded,
                'plot':                 cam,
                'precision':            round(p, 4),
                'recall':               round(r, 4),
                'f1_score':             round(f1, 4),
                'true_positives':       tp,
                'false_positives':      fp,
                'false_negatives':      fn,
                'ground_truth_total':   tp+fn,
            })

        print(f'\n  [Per-class breakdown per plot]')
        print(f'  {"Plot":45}  {"Class":12}  {"Precision":>10}  {"Recall":>7}  {"F1_Score":>9}  {"True_Pos":>9}  {"False_Pos":>10}  {"False_Neg":>10}  {"Accuracy":>9}  {"Correct":>8}  {"Wrong":>6}')
        print(f'  {"-"*130}')

        for cam in sorted(plots.keys()):
            cam_gt    = plots[cam]['gt']
            cam_preds = plots[cam]['preds']
            # Use defaultdict so unknown predicted classes don't cause KeyError
            cls_tp      = _dd12(int)
            cls_fp      = _dd12(int)
            cls_fn      = _dd12(int)
            cls_correct = _dd12(int)
            cls_wrong   = _dd12(int)
            cls_errors  = _dd12(lambda: _dd12(int))

            for img_p, gt_boxes in cam_gt.items():
                insect = [p for p in cam_preds.get(img_p,[]) if not p.get('is_bg')]
                pairs, unp, ung = center_match(insect, gt_boxes)
                for pi,gi in pairs:
                    pc=insect[pi]['cls']; gc=gt_boxes[gi]['cls']
                    if pc==gc: cls_tp[gc]+=1; cls_correct[gc]+=1
                    else: cls_wrong[gc]+=1; cls_errors[gc][pc]+=1; cls_fp[pc]+=1; cls_fn[gc]+=1
                for pi in unp: cls_fp[insect[pi]['cls']]+=1
                for gi in ung: cls_fn[gt_boxes[gi]['cls']]+=1

            first = True
            for c in section_classes:
                tp=cls_tp[c]; fp=cls_fp[c]; fn=cls_fn[c]
                correct=cls_correct[c]; wrong=cls_wrong[c]
                if tp==0 and fp==0 and fn==0: continue
                p=tp/max(1,tp+fp); r=tp/max(1,tp+fn); f1=2*p*r/max(1e-8,p+r)
                acc=correct/max(1,tp)
                cam_label = cam if first else ''
                print(f'  {cam_label:45}  {c:12}  {p:>10.3f}  {r:>7.3f}  {f1:>9.3f}  {tp:>9}  {fp:>10}  {fn:>10}  {acc:>9.3f}  {correct:>8}  {wrong:>6}')
                if cls_errors[c]:
                    err_str = '  '.join(f'{c}\u2192{pc}:{n}' for pc,n in sorted(cls_errors[c].items(), key=lambda x:-x[1]))
                    print(f'  {"":45}  {"  misclassified:":14}  {err_str}')
                first = False
                _xls_plot_cls.append({
                    'pipeline':               label,
                    'bumblebee_excluded':     bumblebee_excluded,
                    'plot':                   cam,
                    'class':                  c,
                    'precision':              round(p,   4),
                    'recall':                 round(r,   4),
                    'f1_score':               round(f1,  4),
                    'true_positives':         tp,
                    'false_positives':        fp,
                    'false_negatives':        fn,
                    'classification_accuracy': round(acc, 4),
                    'correct':                correct,
                    'wrong':                  wrong,
                })
            if not first: print()

_regular_idx = {k: v for k, v in pred_indexes.items() if not k.endswith('_no_bb')}
_no_bb_idx   = {k: v for k, v in pred_indexes.items() if k.endswith('_no_bb')}

print('\n' + '█'*100)
print('  ALL CLASSES  (bumblebee · fly · butterfly · other)')
print('█'*100)
_perplot_section(_regular_idx, GT_CLASSES)

print('\n\n' + '█'*100)
print('  BUMBLEBEE EXCLUDED  (fly · butterfly · other)')
print('█'*100)
_perplot_section(_regular_idx, CLASSES_NO_BB, gt_dict=gt_no_bb, bumblebee_excluded=True)

_stop_capture('per_plot_metrics')


##### Cell 13 — Combined pipeline analysis (crop + YOLO)

For each crop pipeline paired with YOLO, shows:
- How many GT insects each pipeline detects alone
- How many are detected by BOTH (overlap)
- Combined recall (union of detections)
- Among insects detected by both: classification agreement


In [ ]:
_start_capture('combined_analysis')
print('\n' + '='*90)
print('Combined Pipeline Analysis (crop + YOLO)')
print('='*90)

# Auto-detect YOLO label from YOLO_RUNS (configured in Cell 2)
_yolo_labels = list(yolo_run_data.keys())
if not _yolo_labels:
    print('⚠️  No YOLO runs loaded — skipping combined analysis.')
    _stop_capture('combined_analysis')
else:
    YOLO_LABEL = _yolo_labels[0]
    if len(_yolo_labels) > 1:
        print(f'ℹ️  Multiple YOLO runs: {_yolo_labels}. Using: {YOLO_LABEL}')
    yolo_preds = pred_indexes.get(YOLO_LABEL, {})

crop_labels = [l for l in pred_indexes if l != YOLO_LABEL]

for crop_label in crop_labels:
    crop_preds = pred_indexes[crop_label]
    print(f'\n{"="*70}')
    print(f'  {crop_label}  +  {YOLO_LABEL}')
    print(f'{"="*70}')

    # For each GT insect, check who detected it
    crop_only=0; yolo_only=0; both=0; neither=0
    agree=0; disagree=0
    combined_tp=0; combined_fp_crop=0; combined_fp_yolo=0

    # Track per-class combined recall
    cls_gt   = {c:0 for c in GT_CLASSES}
    cls_crop = {c:0 for c in GT_CLASSES}
    cls_yolo = {c:0 for c in GT_CLASSES}
    cls_both = {c:0 for c in GT_CLASSES}
    cls_union= {c:0 for c in GT_CLASSES}

    for img_p, gt_boxes in gt.items():
        if not gt_boxes: continue
        crop_ins = [p for p in crop_preds.get(img_p,[]) if not p.get('is_bg')]
        yolo_ins = yolo_preds.get(img_p, [])

        crop_pairs, _, crop_ung = center_match(crop_ins, gt_boxes)
        yolo_pairs, _, yolo_ung = center_match(yolo_ins, gt_boxes)

        # class-agnostic: which GT indices were spatially found?
        crop_det_gt   = {gi for _, gi in crop_pairs}
        yolo_det_gt   = {gi for _, gi in yolo_pairs}
        # gi → pi reverse maps (for class lookup)
        crop_gi_to_pi = {gi: pi for pi, gi in crop_pairs}
        yolo_gi_to_pi = {gi: pi for pi, gi in yolo_pairs}

        for gi, g in enumerate(gt_boxes):
            gc = g['cls']
            cls_gt[gc] = cls_gt.get(gc,0) + 1
            in_crop = gi in crop_det_gt   # location match only
            in_yolo = gi in yolo_det_gt   # location match only

            # class-specific: detected AND predicted the right class
            _crop_pi = crop_gi_to_pi.get(gi)
            in_crop_cls = _crop_pi is not None and crop_ins[_crop_pi]['cls'] == gc
            _yolo_pi = yolo_gi_to_pi.get(gi)
            in_yolo_cls = _yolo_pi is not None and yolo_ins[_yolo_pi]['cls'] == gc

            # per-class stats: class-specific (detect + correct label)
            if in_crop_cls: cls_crop[gc] = cls_crop.get(gc,0) + 1
            if in_yolo_cls: cls_yolo[gc] = cls_yolo.get(gc,0) + 1
            if in_crop_cls or in_yolo_cls:
                cls_union[gc] = cls_union.get(gc,0) + 1
            if in_crop_cls and in_yolo_cls:
                cls_both[gc] = cls_both.get(gc,0) + 1

            # overall totals: class-agnostic (spatial coverage)
            if in_crop and in_yolo:
                both += 1
                cp = crop_ins[_crop_pi]['cls']
                yp = yolo_ins[_yolo_pi]['cls']
                if cp == yp == gc: agree += 1
                elif cp == gc or yp == gc: agree += 1  # at least one correct
                else: disagree += 1
            elif in_crop:
                crop_only += 1
            elif in_yolo:
                yolo_only += 1
            else:
                neither += 1

    total_gt = crop_only + yolo_only + both + neither
    combined_detected = crop_only + yolo_only + both
    combined_recall = combined_detected / max(1, total_gt)
    crop_recall  = (crop_only + both) / max(1, total_gt)
    yolo_recall  = (yolo_only + both) / max(1, total_gt)

    print(f'\n  GT insects total      : {total_gt}')
    print(f'  Detected by crop only : {crop_only:>4}  ({100*crop_only/max(1,total_gt):.1f}%)')
    print(f'  Detected by YOLO only : {yolo_only:>4}  ({100*yolo_only/max(1,total_gt):.1f}%)')
    print(f'  Detected by BOTH      : {both:>4}  ({100*both/max(1,total_gt):.1f}%)')
    print(f'  Detected by neither   : {neither:>4}  ({100*neither/max(1,total_gt):.1f}%)')
    print(f'')
    print(f'  [Headline totals use class-agnostic detection; per-class rows use class-specific]')
    print(f'  Crop recall alone     : {crop_recall:.3f}  (class-agnostic detection)')
    print(f'  YOLO recall alone     : {yolo_recall:.3f}')
    print(f'  Combined recall (union): {combined_recall:.3f}  (+{100*(combined_recall-max(crop_recall,yolo_recall))/max(0.001,max(crop_recall,yolo_recall)):.1f}% over best single)')
    print(f'')
    print(f'  Among detected by BOTH ({both}):')
    print(f'    Classification agree : {agree}')
    print(f'    Classification disagree: {disagree}')

    print(f'\n  Per-class combined recall:')
    print(f'  {"Class":15}  {"GT":>5}  {"crop":>6}  {"yolo":>6}  {"both":>6}  {"union":>6}  {"R_crop":>8}  {"R_yolo":>8}  {"R_union":>8}')
    print(f'  {"-"*75}')
    for c in GT_CLASSES:
        n  = cls_gt.get(c,0)
        nc = cls_crop.get(c,0)
        ny = cls_yolo.get(c,0)
        nb = cls_both.get(c,0)
        nu = cls_union.get(c,0)
        rc = nc/max(1,n); ry = ny/max(1,n); ru = nu/max(1,n)
        print(f'  {c:15}  {n:>5}  {nc:>6}  {ny:>6}  {nb:>6}  {nu:>6}  {rc:>8.3f}  {ry:>8.3f}  {ru:>8.3f}')

# ── Collect combined analysis for xlsx ─────────────────────────────
_xls_combined = []
for _crop_label in [_l for _l in pred_indexes if _l != YOLO_LABEL]:
    _cp = pred_indexes[_crop_label]; _yp = pred_indexes.get(YOLO_LABEL, {})
    _co=_yo=_bo=_ne=0
    _cg2={c:0 for c in GT_CLASSES}; _cc2={c:0 for c in GT_CLASSES}
    _cy2={c:0 for c in GT_CLASSES}; _cu2={c:0 for c in GT_CLASSES}
    for _img_p, _gt_boxes in gt.items():
        if not _gt_boxes: continue
        _ci=[p for p in _cp.get(_img_p,[]) if not p.get('is_bg')]
        _yi=_yp.get(_img_p,[])
        _cp3,_,_=center_match(_ci,_gt_boxes); _yp3,_,_=center_match(_yi,_gt_boxes)
        _cdet={gi for _,gi in _cp3}; _ydet={gi for _,gi in _yp3}
        _cmap={gi:pi for pi,gi in _cp3}; _ymap={gi:pi for pi,gi in _yp3}
        for _gi,_g in enumerate(_gt_boxes):
            _gc=_g['cls']; _cg2[_gc]=_cg2.get(_gc,0)+1
            _ic_det=_gi in _cdet; _iy_det=_gi in _ydet
            _ic_cls=(_cmap.get(_gi) is not None and _ci[_cmap[_gi]]['cls']==_gc)
            _iy_cls=(_ymap.get(_gi) is not None and _yi[_ymap[_gi]]['cls']==_gc)
            if _ic_cls: _cc2[_gc]=_cc2.get(_gc,0)+1
            if _iy_cls: _cy2[_gc]=_cy2.get(_gc,0)+1
            if _ic_cls or _iy_cls: _cu2[_gc]=_cu2.get(_gc,0)+1
            if _ic_det and _iy_det: _bo+=1
            elif _ic_det: _co+=1
            elif _iy_det: _yo+=1
            else: _ne+=1
    _tot=_co+_yo+_bo+_ne
    # summary row
    _xls_combined.append({'crop_pipeline':_crop_label,'yolo_pipeline':YOLO_LABEL,
        'class':'__ALL__','total_gt':_tot,
        'only_crop':_co,'only_yolo':_yo,'detected_by_both':_bo,'detected_by_neither':_ne,
        'crop_recall':round((_co+_bo)/max(1,_tot),4),
        'yolo_recall':round((_yo+_bo)/max(1,_tot),4),
        'combined_recall':round((_co+_yo+_bo)/max(1,_tot),4)})
    for _c in GT_CLASSES:
        _n=_cg2.get(_c,0)
        _xls_combined.append({'crop_pipeline':_crop_label,'yolo_pipeline':YOLO_LABEL,
            'class':_c,'total_gt':_n,
            'crop_only':0,'yolo_only':0,'both':0,'neither':0,
            'recall_crop':round(_cc2.get(_c,0)/max(1,_n),4),
            'recall_yolo':round(_cy2.get(_c,0)/max(1,_n),4),
            'recall_union':round(_cu2.get(_c,0)/max(1,_n),4)})

_stop_capture('combined_analysis')


##### Cell 14 — Unique detections per pipeline

For each pipeline: GT insects that ONLY this pipeline detected, no other pipeline found them.
Useful for understanding what each pipeline uniquely contributes.


In [ ]:
print('\n' + '='*70)
print('Unique Detections — insects only this pipeline found')
print('='*70)

all_labels = list(pred_indexes.keys())

# For each GT insect, record which pipelines detected it (class-agnostic location match).
# Note: 'detected' here means any bbox overlapped the GT location, regardless of
# predicted class. This measures spatial coverage, not joint detection+classification.
gt_detected_by = {}  # (img_p, gi) -> set of pipeline labels

for label, preds_by_img in pred_indexes.items():
    for img_p, gt_boxes in gt.items():
        if not gt_boxes: continue
        insect = [p for p in preds_by_img.get(img_p,[]) if not p.get('is_bg')]
        pairs, _, _ = center_match(insect, gt_boxes)
        for _, gi in pairs:
            key = (img_p, gi)
            gt_detected_by.setdefault(key, set()).add(label)

# For each pipeline, count unique detections
print(f'\n  {"Pipeline":40}  {"Unique":>7}  {"% of GT":>8}  {"% of its TP":>12}')
print(f'  {"-"*70}')

total_gt = sum(1 for boxes in gt.values() for _ in boxes)

for label in all_labels:
    # TP for this pipeline
    tp_keys = {k for k,v in gt_detected_by.items() if label in v}
    # Unique: only this pipeline detected it
    unique_keys = {k for k,v in gt_detected_by.items()
                   if v == {label}}
    n_unique = len(unique_keys)
    n_tp     = len(tp_keys)
    pct_gt   = 100*n_unique/max(1,total_gt)
    pct_tp   = 100*n_unique/max(1,n_tp)
    print(f'  {label:40}  {n_unique:>7}  {pct_gt:>7.1f}%  {pct_tp:>11.1f}%')

# Per-class breakdown
print(f'\n  Per-class unique detections:')
print(f'  {"Pipeline":40}  ' + '  '.join(f'{c:>10}' for c in GT_CLASSES))
print(f'  {"-"*80}')

for label in all_labels:
    unique_keys = {k for k,v in gt_detected_by.items() if v == {label}}
    cls_counts = {c:0 for c in GT_CLASSES}
    for (img_p, gi) in unique_keys:
        gc = gt[img_p][gi]['cls']
        cls_counts[gc] = cls_counts.get(gc,0) + 1
    row = f'  {label:40}  ' + '  '.join(f'{cls_counts.get(c,0):>10}' for c in GT_CLASSES)
    print(row)

# Also show: insects detected by ALL pipelines
all_detected = {k for k,v in gt_detected_by.items() if len(v) == len(all_labels)}
print(f'\n  Detected by ALL {len(all_labels)} pipelines: {len(all_detected)} GT insects')
print(f'  Not detected by ANY pipeline: {total_gt - len(gt_detected_by)} GT insects')

_start_capture('unique_detections')
# ── Collect unique detections for xlsx ───────────────────────────
_xls_unique = []
for _label in all_labels:
    _tp_k = {k for k,v in gt_detected_by.items() if _label in v}
    _uq_k = {k for k,v in gt_detected_by.items() if v == {_label}}
    _cls_uq = {c:0 for c in GT_CLASSES}
    for (_ip,_gi) in _uq_k:
        _gc=gt[_ip][_gi]['cls']; _cls_uq[_gc]=_cls_uq.get(_gc,0)+1
    _row={'pipeline':_label,'uniquely_detected':len(_uq_k),
          'pipeline_true_positives':len(_tp_k),'total_ground_truth':total_gt,
          'pct_of_ground_truth':round(100*len(_uq_k)/max(1,total_gt),2),
          'pct_of_true_positives':round(100*len(_uq_k)/max(1,len(_tp_k)),2)}
    for _c in GT_CLASSES: _row[f'unique_{_c}']=_cls_uq.get(_c,0)
    _xls_unique.append(_row)

_stop_capture('unique_detections')
# ── Write everything to one xlsx ──────────────────────────────────────────
try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','install','openpyxl','-q'])
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

def _write_sheet(ws, rows, col_names=None):
    """Write a list-of-dicts to a worksheet with header row."""
    if not rows: return
    cols = col_names or list(rows[0].keys())
    HDR_FILL = PatternFill('solid', start_color='1F4E79')
    HDR_FONT = Font(bold=True, color='FFFFFF', name='Arial', size=10)
    ROW_FONT = Font(name='Arial', size=10)
    ALT_FILL = PatternFill('solid', start_color='D6E4F7')

    for ci, col in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=col)
        cell.font = HDR_FONT; cell.fill = HDR_FILL
        cell.alignment = Alignment(horizontal='center')

    for ri, row in enumerate(rows, 2):
        for ci, col in enumerate(cols, 1):
            cell = ws.cell(row=ri, column=ci, value=row.get(col,''))
            cell.font = ROW_FONT
            if ri % 2 == 0: cell.fill = ALT_FILL

    # Auto-width (capped at 40)
    for ci, col in enumerate(cols, 1):
        max_w = max(len(str(col)), max((len(str(r.get(col,''))) for r in rows), default=0))
        ws.column_dimensions[get_column_letter(ci)].width = min(max_w + 3, 40)

    ws.freeze_panes = 'A2'

wb = Workbook()

# Sheet 0: README
ws0 = wb.active; ws0.title = 'README'
_readme = [
    ('Sheet', 'Contents'),
    ('overall_metrics',   'Overall detection P/R/F1 per pipeline (+ FN breakdown, classification accuracy)'),
    ('per_class',         'P/R/F1 per pipeline x class'),
    ('confusion_matrix',  'GT class vs predicted class counts (incl. bg_rejected and missed)'),
    ('threshold',         'Precision/Recall/F1 at every confidence threshold per pipeline'),
    ('per_plot_det',      'Class-agnostic detection metrics per camera folder per pipeline'),
    ('per_plot_cls',      'Per-class metrics per camera folder per pipeline'),
    ('combined',          'Crop + YOLO union recall analysis per class'),
    ('unique_detections', 'GT insects only detected by each pipeline exclusively'),
    ('raw_detections',    'Every bbox row used in evaluation (tp/fp/fn_*)'),
]
_write_sheet(ws0, [{'Sheet':s,'Contents':c} for s,c in _readme[1:]], ['Sheet','Contents'])
ws0['A1'].value = 'Sheet'; ws0['B1'].value = 'Contents'  # already set by _write_sheet

# Sheets 1-9
_sheets = [
    ('overall_metrics',   _xls_overall),
    ('per_class',         _xls_per_class),
    ('confusion_matrix',  _xls_confusion),
    ('threshold',         _xls_threshold),
    ('per_plot_det',      _xls_plot_det),
    ('per_plot_cls',      _xls_plot_cls),
    ('combined',          _xls_combined),
    ('unique_detections', _xls_unique),
    ('raw_detections',    _xls_raw),
]
for name, data in _sheets:
    _ws = wb.create_sheet(name)
    _write_sheet(_ws, data)

_xlsx_path = EVAL_DIR / f'eval_{RUN_TS}.xlsx'
wb.save(str(_xlsx_path))

# ── Write captured print output to report.txt ────────────────────────
_report = {'run_timestamp': RUN_TS, 'sections': _report_sections}
(EVAL_DIR / 'report.json').write_text(_json.dumps(_report, indent=2, ensure_ascii=False))

print(f'\n\u2713 Saved to {EVAL_DIR}/')
print(f'   └ eval_{RUN_TS}.xlsx   ← all metrics (9 sheets)')
print(f'   └ report.json         ← text output split by section')
print(f'   └ threshold_analysis.png')
print(f'   Sheets: {[s for s,_ in _sheets]}')
print(f'   (Run Cell 15 to append conf_sweep + binary_fn + binary_thr sheets)')
# ── Write outputs_index.json — lists every file saved this run ───────────────
import json as _ji, os as _os

def _eval_dir_files():
    """List all files currently in EVAL_DIR."""
    entries = []
    for p in sorted(EVAL_DIR.iterdir()):
        if p.is_file():
            entries.append({
                'filename': p.name,
                'size_kb':  round(p.stat().st_size / 1024, 1),
                'path':     str(p),
            })
    return entries

_outputs_index = {
    'eval_timestamp': RUN_TS,
    'eval_dir': str(EVAL_DIR),
    'crop_runs': {name: str(path) for name, path in CROP_RUNS.items()},
    'yolo_runs': {name: str(path) for name, path in YOLO_RUNS.items()},
    'files': [
        {'filename': f'eval_{RUN_TS}.xlsx',         'description': 'All metrics — 9 sheets (overall, per-class, confusion, threshold, per-plot, combined, unique, raw)'},
        {'filename': 'report.json',                  'description': 'Full text output of every evaluation section, split by name'},
        {'filename': 'eval_config.json',             'description': 'Run configuration — which inference runs were evaluated, conf thresholds, GT classes'},
        {'filename': 'threshold_analysis.png',       'description': 'PR curve + recall/precision vs confidence threshold for every pipeline (DPI=300)'},
        {'filename': 'operating_curve.png',          'description': 'Recall vs false-positive count at varying thresholds, per pipeline (DPI=300)'},
        {'filename': 'complementarity.png',          'description': 'Crop vs YOLO detection complementarity bar chart per class (DPI=300)'},
        {'filename': 'failure_panel_<pipeline>.png', 'description': 'Qualitative failure panels — images with TP/FP/FN boxes drawn (one file per pipeline, DPI=150)'},
        {'filename': 'fn_crops/',                    'description': 'Directory: image crops around each false-negative GT insect'},
    ],
    'actual_files': _eval_dir_files(),
}

(EVAL_DIR / 'outputs_index.json').write_text(
    _ji.dumps(_outputs_index, indent=2, ensure_ascii=False)
)

print(f'\n✓ outputs_index.json written → {EVAL_DIR / "outputs_index.json"}')
print(f'  Files in EVAL_DIR so far:')
for _f in _outputs_index['actual_files']:
    print(f'    {_f["filename"]:45}  {_f["size_kb"]:>8.1f} KB')



##### Cell 15 — Confidence threshold analysis for five_class pipelines

Answers three questions:
1. **Are FP predictions lower-confidence than TP?** (TP/FP confidence distributions)
2. **How much does each threshold cut FPs vs lose TPs?** (sweep table)
3. **Where do binary-stage FNs come from?** (binary FN breakdown per stage)

All numbers are computed from `results.csv` + CVAT ground truth — nothing is hard-coded.
Re-run this cell after any new inference run to get updated tables.

**Dependency:** Cell 4 (GT loaded into `gt`), Cell 5 (crop rows loaded into `crop_run_data`).

In [ ]:
_start_capture('conf_threshold_analysis')
import numpy as np
from collections import defaultdict, Counter

# ── helpers ──────────────────────────────────────────────────────────────────

def _center_match(px1, py1, px2, py2, gx1, gy1, gx2, gy2, overlap_thresh=0.20):
    """True if pred and GT overlap by center-in-box or area criterion."""
    gcx = (gx1+gx2)/2; gcy = (gy1+gy2)/2
    pcx = (px1+px2)/2; pcy = (py1+py2)/2
    iw  = max(0, min(px2,gx2) - max(px1,gx1))
    ih  = max(0, min(py2,gy2) - max(py1,gy1))
    gt_area = max(1, (gx2-gx1)*(gy2-gy1))
    return ((px1<=gcx<=px2 and py1<=gcy<=py2) or
            (gx1<=pcx<=gx2 and gy1<=pcy<=gy2) or
            (iw*ih/gt_area) >= overlap_thresh)


def _tp_fp_confs(rows, pipe_name, gt):
    """
    Return (tp_confs, fp_confs) for a five_class pipeline.
    TP  = predicted non-background AND overlaps a GT box.
    FP  = predicted non-background AND no GT overlap.
    Uses highest-conf prediction per GT box (greedy match, desc conf order).
    """
    # Build per-image detections: (conf, x1,y1,x2,y2)
    preds_by_img = defaultdict(list)
    p = pipe_name + '__'
    for r in rows:
        if r.get('pollinator_detected') != 'yes': continue
        pt = r.get(p+'pollinator_type', '')
        if not pt or pt == 'background': continue
        try:
            bx=float(r['bbox_x']); by=float(r['bbox_y'])
            bw=float(r['bbox_w']); bh=float(r['bbox_h'])
            conf=float(r.get(p+'group_conf') or 0)
        except (ValueError, KeyError): continue
        img_p = r['_img_path']
        preds_by_img[img_p].append((conf, bx, by, bx+bw, by+bh))

    tp_confs, fp_confs = [], []
    for img_p, preds in preds_by_img.items():
        gt_boxes = gt.get(img_p, [])
        matched_gt = set()
        # Sort by conf desc so high-confidence predictions claim GT first
        for conf, px1,py1,px2,py2 in sorted(preds, key=lambda x: -x[0]):
            is_tp = False
            for gi, g in enumerate(gt_boxes):
                if gi in matched_gt: continue
                if _center_match(px1,py1,px2,py2, g['x1'],g['y1'],g['x2'],g['y2']):
                    matched_gt.add(gi); is_tp = True; break
            (tp_confs if is_tp else fp_confs).append(conf)
    return np.array(tp_confs), np.array(fp_confs)


def _binary_fn_analysis(rows, gt):
    """
    For two_stage pipeline: classify each crop that was rejected as background.
    Returns (fn_confs, true_bg_confs) where fn = true GT insect rejected.
    Also returns the three-stage FN breakdown across all GT insects.
    """
    p = 'two_stage__'

    # Mark which GT boxes were ever overlapped by ANY crop (seen by motion),
    # and which were passed by the binary classifier.
    # Use mutable dicts so we can set flags.
    gt_status = {}   # (img_p, gi) -> {'seen': bool, 'passed_binary': bool}
    for img_p, boxes in gt.items():
        for gi in range(len(boxes)):
            gt_status[(img_p, gi)] = {'seen': False, 'passed_binary': False}

    fn_confs = []       # p_background for true insects binary rejected
    true_bg_confs = []  # p_background for correctly rejected background

    for r in rows:
        if r.get('pollinator_detected') != 'yes': continue
        try:
            bx=float(r['bbox_x']); by=float(r['bbox_y'])
            bw=float(r['bbox_w']); bh=float(r['bbox_h'])
        except: continue
        x1,y1,x2,y2 = bx, by, bx+bw, by+bh
        img_p = r['_img_path']
        bl = r.get(p+'binary_label', '')
        try: bconf = float(r.get(p+'binary_conf') or 0)
        except: bconf = 0.0

        matched_gi = None
        for gi, g in enumerate(gt.get(img_p, [])):
            if _center_match(x1,y1,x2,y2, g['x1'],g['y1'],g['x2'],g['y2']):
                matched_gi = gi; break

        # Mark seen by motion
        if matched_gi is not None:
            gt_status[(img_p, matched_gi)]['seen'] = True
            if bl == 'insect':
                gt_status[(img_p, matched_gi)]['passed_binary'] = True

        # Collect binary confidence for bg-rejected crops
        if bl == 'background':
            if matched_gi is not None:
                fn_confs.append(bconf)
            else:
                true_bg_confs.append(bconf)

    never_seen       = sum(1 for v in gt_status.values() if not v['seen'])
    binary_rejected  = sum(1 for v in gt_status.values() if v['seen'] and not v['passed_binary'])
    passed_binary    = sum(1 for v in gt_status.values() if v['passed_binary'])
    total_gt         = len(gt_status)

    return (np.array(fn_confs), np.array(true_bg_confs),
            {'total_gt': total_gt, 'never_seen': never_seen,
             'binary_rejected': binary_rejected, 'passed_binary': passed_binary})


# ═══════════════════════════════════════════════════════════════════════════
# PART 1 — TP vs FP confidence distributions for five_class pipelines
# ═══════════════════════════════════════════════════════════════════════════

print('\n' + '='*65)
print('Part 1 — TP vs FP confidence distributions (five_class pipelines)')
print('='*65)

_five_class_pipes = [
    (run_name, pipe_name)
    for run_name, run_data in crop_run_data.items()
    for pipe_name in run_data['pipes']
    if 'five_class' in pipe_name
]

tp_fp_data = {}   # pipe_label -> (tp_arr, fp_arr)

for run_name, pipe_name in _five_class_pipes:
    rows = crop_run_data[run_name]['rows']
    lbl  = f'{run_name}/{pipe_name}'
    tp, fp = _tp_fp_confs(rows, pipe_name, gt)
    tp_fp_data[lbl] = (tp, fp)
    print(f'\n  {lbl}')
    print(f'  {"-"*55}')
    if len(tp):
        print(f'  TP ({len(tp):5d} crops)  conf: ' +
              f'min={tp.min():.3f}  p25={np.percentile(tp,25):.3f}  ' +
              f'median={np.median(tp):.3f}  p75={np.percentile(tp,75):.3f}  max={tp.max():.3f}')
    if len(fp):
        print(f'  FP ({len(fp):5d} crops)  conf: ' +
              f'min={fp.min():.3f}  p25={np.percentile(fp,25):.3f}  ' +
              f'median={np.median(fp):.3f}  p75={np.percentile(fp,75):.3f}  max={fp.max():.3f}')

# Summary table
if tp_fp_data:
    print(f'\n  {"Pipeline":35}  {"TP median":>10}  {"FP median":>10}  {"TP count":>9}  {"FP count":>9}')
    print(f'  {"-"*80}')
    for lbl, (tp, fp) in tp_fp_data.items():
        tm = f'{np.median(tp):.3f}' if len(tp) else '—'
        fm = f'{np.median(fp):.3f}' if len(fp) else '—'
        print(f'  {lbl:35}  {tm:>10}  {fm:>10}  {len(tp):>9}  {len(fp):>9}')

# ── Collect Part 1 data for xlsx ─────────────────────────────────────────
_xls_conf_sweep = []   # Part 2 threshold rows
_xls_tp_fp_dist = []   # Part 1 distribution summary
for _lbl, (_tp, _fp) in tp_fp_data.items():
    _xls_tp_fp_dist.append({
        'pipeline': _lbl,
        'tp_count': len(_tp),
        'tp_min':    round(float(_tp.min()),   3) if len(_tp) else None,
        'tp_p25':    round(float(np.percentile(_tp, 25)), 3) if len(_tp) else None,
        'tp_median': round(float(np.median(_tp)), 3) if len(_tp) else None,
        'tp_p75':    round(float(np.percentile(_tp, 75)), 3) if len(_tp) else None,
        'tp_max':    round(float(_tp.max()),   3) if len(_tp) else None,
        'fp_count': len(_fp),
        'fp_min':    round(float(_fp.min()),   3) if len(_fp) else None,
        'fp_p25':    round(float(np.percentile(_fp, 25)), 3) if len(_fp) else None,
        'fp_median': round(float(np.median(_fp)), 3) if len(_fp) else None,
        'fp_p75':    round(float(np.percentile(_fp, 75)), 3) if len(_fp) else None,
        'fp_max':    round(float(_fp.max()),   3) if len(_fp) else None,
    })

# ═══════════════════════════════════════════════════════════════════════════
# PART 2 — Threshold sweep: FP removed vs TP lost
# ═══════════════════════════════════════════════════════════════════════════

print('\n' + '='*65)
print('Part 2 — Threshold sweep (eval-time conf_threshold)')
print('='*65)
print('  Threshold = minimum confidence to count a five_class prediction as a detection.')
print('  Predictions below threshold are treated as background.')

for lbl, (tp, fp) in tp_fp_data.items():
    if not len(tp): continue
    total_tp = len(tp); total_fp = len(fp)
    print(f'\n  {lbl}')
    print(f'  {"Threshold":>10}  {"TP kept":>8}  {"TP recall":>10}  {"FP kept":>8}  {"FP removed":>11}  {"Precision":>10}')
    print(f'  {"-"*70}')
    _pipe_name = lbl.split('/')[-1]
    for thr in [0.0, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]:
        tp_k = int((tp >= thr).sum()); fp_k = int((fp >= thr).sum())
        fp_rm = total_fp - fp_k
        prec = tp_k/max(1, tp_k+fp_k)
        rec  = tp_k/max(1, total_tp)
        print(f'  {thr:>10.2f}  {tp_k:>8}  {rec:>9.1%}  {fp_k:>8}  {fp_rm:>10}  ({100*fp_rm/max(1,total_fp):4.0f}%)  {prec:>9.3f}')
        _xls_conf_sweep.append({
            'pipeline':            lbl,
            'threshold':           thr,
            'tp_kept':             tp_k,
            'tp_recall_pct':       round(100*rec, 1),
            'fp_kept':             fp_k,
            'fp_removed':          fp_rm,
            'fp_removed_pct':      round(100*fp_rm/max(1,total_fp), 1),
            'precision':           round(prec, 3),
            'current_threshold':   CONF_THRESHOLDS.get(_pipe_name, 'not set'),
        })
    print(f'  (Current CONF_THRESHOLDS setting for this pipeline: ' +
          f'{CONF_THRESHOLDS.get(_pipe_name, "not set")!r})')

# ═══════════════════════════════════════════════════════════════════════════
# PART 3 — Binary stage FN breakdown (two_stage pipeline)
# ═══════════════════════════════════════════════════════════════════════════

print('\n' + '='*65)
print('Part 3 — two_stage binary stage FN analysis')
print('='*65)

_two_stage_runs = [
    (run_name, pipe_name)
    for run_name, run_data in crop_run_data.items()
    for pipe_name in run_data['pipes']
    if 'two_stage' in pipe_name
]

_xls_binary_fn = []   # Part 3 stage breakdown rows
_xls_binary_thr = []  # Part 3 binary threshold sweep rows

for run_name, pipe_name in _two_stage_runs:
    rows = crop_run_data[run_name]['rows']
    lbl  = f'{run_name}/{pipe_name}'
    fn_confs, bg_confs, stages = _binary_fn_analysis(rows, gt)
    total = stages['total_gt']

    print(f'\n  {lbl}')
    print(f'  {"─"*60}')
    print(f'  Total GT insects:                       {total:5}')
    print(f"  ✗ Never cropped (motion detection miss):{stages['never_seen']:5}  " +
          f"({100*stages['never_seen']/total:.0f}%)  ← threshold cannot recover")
    print(f"  ✗ Cropped but binary said background:   {stages['binary_rejected']:5}  " +
          f"({100*stages['binary_rejected']/total:.0f}%)  ← see threshold analysis below")
    print(f"  ✓ Passed binary (reach group clf):      {stages['passed_binary']:5}  " +
          f"({100*stages['passed_binary']/total:.0f}%)")
    print()

    _xls_binary_fn.append({
        'pipeline':               lbl,
        'total_gt':               total,
        'never_seen_motion_miss': stages['never_seen'],
        'never_seen_pct':         round(100*stages['never_seen']/max(1,total), 1),
        'binary_rejected_fn':     stages['binary_rejected'],
        'binary_rejected_pct':    round(100*stages['binary_rejected']/max(1,total), 1),
        'passed_binary':          stages['passed_binary'],
        'passed_binary_pct':      round(100*stages['passed_binary']/max(1,total), 1),
        'fn_insect_count':        len(fn_confs),
        'fn_median_p_bg':         round(float(np.median(fn_confs)), 3) if len(fn_confs) else None,
        'true_bg_count':          len(bg_confs),
        'true_bg_median_p_bg':    round(float(np.median(bg_confs)), 3) if len(bg_confs) else None,
    })

    if len(fn_confs):
        print(f'  p_background distribution for TRUE INSECTS that binary rejected:')
        print(f'    min={fn_confs.min():.3f}  p25={np.percentile(fn_confs,25):.3f}  ' +
              f'median={np.median(fn_confs):.3f}  p75={np.percentile(fn_confs,75):.3f}')
        print(f'  p_background for true background (correctly rejected):')
        print(f'    min={bg_confs.min():.3f}  p25={np.percentile(bg_confs,25):.3f}  ' +
              f'median={np.median(bg_confs):.3f}  p75={np.percentile(bg_confs,75):.3f}')
        print()
        print(f'  *** KEY FINDING: distributions are nearly identical — threshold')
        print(f'  *** adjustment alone cannot reliably separate FN insects from true BG.')
        print()
        print(f'  Threshold sweep: predict insect if p_background < threshold')
        print(f'  (group classifier has NO background class — leaked BG becomes FP)')
        print(f'  {"p_bg <":>10}  {"FN recovered":>13}  {"Recover %":>10}  {"True BG leaked":>15}  {"BG/FN ratio":>12}')
        print(f'  {"-"*68}')
        for thr in [0.99, 0.97, 0.95, 0.90, 0.80, 0.70, 0.60, 0.51]:
            fn_rec = int((fn_confs  < thr).sum())
            bg_lk  = int((bg_confs  < thr).sum())
            ratio  = bg_lk/max(1, fn_rec)
            print(f'  {thr:>10.2f}  {fn_rec:>13}  ' +
                  f'{100*fn_rec/max(1,len(fn_confs)):>9.0f}%  {bg_lk:>15}  {ratio:>10.0f}:1')
            _xls_binary_thr.append({
                'pipeline':        lbl,
                'p_bg_threshold':  thr,
                'fn_recovered':    fn_rec,
                'fn_recover_pct':  round(100*fn_rec/max(1,len(fn_confs)), 1),
                'true_bg_leaked':  bg_lk,
                'bg_fn_ratio':     round(ratio, 1),
            })

print(f'\n  Summary: to reduce FNs, tune motion detection (darker_threshold, min_contour_area)')
print(f'  or switch to YOLO which does not depend on motion detection at all.')

_stop_capture('conf_threshold_analysis')

# ── Append conf_sweep + binary_fn sheets to existing xlsx ────────────────
try:
    from openpyxl import load_workbook as _lw2
    _wb2 = _lw2(str(_xlsx_path))
    # Append README entries
    _ws_rm = _wb2['README']
    _rn = _ws_rm.max_row + 1
    for _nm, _desc in [
        ('tp_fp_dist',  'Five-class TP vs FP confidence distributions per pipeline (Part 1)'),
        ('conf_sweep',  'Eval-time confidence threshold sweep: TP recall vs FP removed (Part 2)'),
        ('binary_fn',   'Two-stage binary FN stage breakdown per pipeline (Part 3)'),
        ('binary_thr',  'Two-stage binary p_background threshold sweep — FN recovery vs BG leak (Part 3)'),
    ]:
        _ws_rm.cell(row=_rn, column=1, value=_nm)
        _ws_rm.cell(row=_rn, column=2, value=_desc)
        _rn += 1
    # Add sheets
    for _sname, _sdata in [
        ('tp_fp_dist', _xls_tp_fp_dist),
        ('conf_sweep',  _xls_conf_sweep),
        ('binary_fn',   _xls_binary_fn),
        ('binary_thr',  _xls_binary_thr),
    ]:
        _ws_new = _wb2.create_sheet(_sname)
        _write_sheet(_ws_new, _sdata)
    _wb2.save(str(_xlsx_path))
    print(f'\n✓ Appended 4 sheets to {_xlsx_path.name}: tp_fp_dist, conf_sweep, binary_fn, binary_thr')
except Exception as _e:
    print(f'\n⚠ Could not update xlsx: {_e}')

# ── Re-write report.json with all sections including this one ─────────────
try:
    _report = {'run_timestamp': RUN_TS, 'sections': _report_sections}
    (EVAL_DIR / 'report.json').write_text(_json.dumps(_report, indent=2, ensure_ascii=False))
    print(f'✓ Updated report.json  (sections: {list(_report_sections.keys())})')
except Exception as _e:
    print(f'\n⚠ Could not update report.json: {_e}')


##### Cell 16 — Distribution Shift Analysis

Compares pipeline performance across **image quality subsets** (e.g. clear vs foggy/dusty/backlit).
Useful for understanding how sensor/environment conditions affect recall.

**Two ways to define subsets:**
1. **Manual list** — paste filenames or camera folders into `DEGRADED_CAMERAS` / `DEGRADED_IMAGES`
2. **Auto-detect** — set `AUTO_DETECT=True` to flag images by low contrast (proxy for fog/dust)

Reuses `evaluate_one_pipeline()` from Cell 6 — no new metrics logic needed.
Run Cell 8 first so `pred_indexes`, `gt`, and `all_results` are available.

In [ ]:
_start_capture('distribution_shift')
import numpy as np
import cv2
from pathlib import Path
from collections import defaultdict

# ══════════════════════════════════════════════════════════════════════════
#  CONFIG — edit this section
# ══════════════════════════════════════════════════════════════════════════

# Option 1: manually list camera folders that have quality issues
# e.g. ['102_WSCT', '103_WSCT'] — any image under these folders = degraded
DEGRADED_CAMERAS = []   # ← fill in, or leave [] to use auto-detect

# Option 2: manually list specific image filenames (stem only, no extension)
DEGRADED_IMAGES  = []   # ← e.g. ['WSCT1529', 'WSCT1530']

# Option 3: auto-detect low-contrast images (proxy for fog / dust / glare)
AUTO_DETECT         = True   # set False if using manual lists above
AUTO_CONTRAST_LOW   = 30     # Laplacian variance below this → "degraded"
AUTO_CONTRAST_HIGH  = 200    # above this → "clear" (middle = ambiguous, excluded)

# ══════════════════════════════════════════════════════════════════════════

def laplacian_variance(img_path):
    """Sharpness proxy: low value = blurry/foggy/dusty, high = clear."""
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    return float(cv2.Laplacian(img, cv2.CV_64F).var())

# ── Build clear / degraded image sets ─────────────────────────────────────
all_img_paths = sorted(gt.keys())

if DEGRADED_CAMERAS or DEGRADED_IMAGES:
    degraded = set()
    for p in all_img_paths:
        pp = Path(p)
        if pp.parent.name in DEGRADED_CAMERAS:
            degraded.add(p)
        if pp.stem in DEGRADED_IMAGES:
            degraded.add(p)
    clear = set(all_img_paths) - degraded
    method = 'manual'
else:
    # Auto-detect via Laplacian variance
    print('Auto-detecting image quality (Laplacian variance)...')
    scores = {}
    for p in all_img_paths:
        v = laplacian_variance(p)
        if v is not None:
            scores[p] = v

    clear    = {p for p, v in scores.items() if v >= AUTO_CONTRAST_HIGH}
    degraded = {p for p, v in scores.items() if v <  AUTO_CONTRAST_LOW}
    ambig    = set(scores) - clear - degraded
    method   = f'auto (Laplacian: low<{AUTO_CONTRAST_LOW}, high≥{AUTO_CONTRAST_HIGH})'
    print(f'  Clear: {len(clear)}  Degraded: {len(degraded)}  Ambiguous (excluded): {len(ambig)}')
    if scores:
        vals = list(scores.values())
        print(f'  Contrast range: {min(vals):.1f} – {max(vals):.1f}  median={np.median(vals):.1f}')

print(f'\nSubset method : {method}')
print(f'Clear images  : {len(clear)}')
print(f'Degraded images: {len(degraded)}')

if len(clear) < 5 or len(degraded) < 5:
    print('\n⚠️  Too few images in one subset — adjust thresholds or use manual lists.')
else:
    gt_clear    = {p: gt[p] for p in clear    if p in gt}
    gt_degraded = {p: gt[p] for p in degraded if p in gt}

    # Per-camera breakdown of degraded images
    cam_counts = defaultdict(int)
    for p in degraded:
        cam_counts[Path(p).parent.name] += 1
    if cam_counts:
        print('\nDegraded images per camera:')
        for cam, n in sorted(cam_counts.items(), key=lambda x: -x[1]):
            print(f'  {cam:40}  {n:>4} images')

    # ── Run evaluation on each subset ─────────────────────────────────────
    print('\n' + '='*80)
    print('Distribution Shift Analysis — Clear vs Degraded')
    print('='*80)

    header = f"  {'Pipeline':30}  {'Subset':10}  {'GT':>5}  {'Recall':>8}  {'Precision':>10}  {'F1':>7}  {'MacroF1':>8}"
    print(f'\n{header}')
    print(f'  {"-"*80}')

    _xls_shift = []
    for label, preds_by_img in pred_indexes.items():
        for subset_name, gt_sub in [('clear', gt_clear), ('degraded', gt_degraded)]:
            preds_sub = {p: preds_by_img[p] for p in gt_sub if p in preds_by_img}
            if not gt_sub:
                continue
            r = evaluate_one_pipeline(preds_sub, gt_sub, GT_CLASSES,
                                      f'{label}_{subset_name}')
            det = r.get('detection', {})
            tp  = det.get('tp', 0); fp = det.get('fp', 0); fn = det.get('fn', 0)
            rec = tp / max(1, tp + fn)
            pre = tp / max(1, tp + fp)
            f1  = 2*pre*rec / max(1e-8, pre+rec)
            n_gt = sum(len(v) for v in gt_sub.values())
            mf1 = r.get('macro_f1', 0)
            print(f"  {label:30}  {subset_name:10}  {n_gt:>5}  {rec:>8.3f}  {pre:>10.3f}  {f1:>7.3f}  {mf1:>8.3f}")
            _xls_shift.append({
                'pipeline': label, 'subset': subset_name,
                'n_gt': n_gt, 'recall': round(rec,4),
                'precision': round(pre,4), 'f1': round(f1,4),
                'macro_f1': round(mf1,4), 'tp': tp, 'fp': fp, 'fn': fn,
            })

    # ── Recall drop summary ────────────────────────────────────────────────
    print(f'\n  {"Pipeline":30}  {"R_clear":>8}  {"R_degraded":>11}  {"Drop":>6}')
    print(f'  {"-"*58}')
    by_pipe = defaultdict(dict)
    for row in _xls_shift:
        by_pipe[row["pipeline"]][row["subset"]] = row["recall"]
    for pipe, subs in sorted(by_pipe.items()):
        rc = subs.get("clear", float("nan"))
        rd = subs.get("degraded", float("nan"))
        drop = rc - rd if rc == rc and rd == rd else float("nan")
        flag = "  ⚠️" if drop > 0.1 else ""
        print(f'  {pipe:30}  {rc:>8.3f}  {rd:>11.3f}  {drop:>+6.3f}{flag}')

_stop_capture('distribution_shift')


##### Cell 17 — Matching Method Comparison: center_match vs IoU ≥ 0.5

Compares two bbox-matching strategies used to align predictions with ground truth:

- **center_match** (3-criteria): a GT–pred pair counts as a match if the GT center
  falls inside the pred box, the pred center falls inside the GT box, or ≥ 20 % of
  the GT area is covered.  Lenient — designed to tolerate SAHI-slice edge overlap.
- **IoU ≥ 0.5**: standard COCO criterion.  Stricter — requires at least 50 %
  intersection-over-union.  Greedy: highest-IoU pairs are claimed first.

Showing both reveals how sensitive reported metrics are to the choice of matcher.

In [ ]:
from collections import defaultdict
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── IoU ≥ threshold matcher ───────────────────────────────────────────────────
def iou_match(pred_boxes, gt_boxes, iou_thresh=0.50):
    """Greedy IoU matching; highest-IoU pairs claimed first."""
    def _iou(p, g):
        ix1 = max(p['x1'], g['x1']); iy1 = max(p['y1'], g['y1'])
        ix2 = min(p['x2'], g['x2']); iy2 = min(p['y2'], g['y2'])
        inter = max(0, ix2-ix1) * max(0, iy2-iy1)
        union = ((p['x2']-p['x1'])*(p['y2']-p['y1']) +
                 (g['x2']-g['x1'])*(g['y2']-g['y1']) - inter)
        return inter / max(1, union)
    candidates = sorted(
        (-_iou(p, g), pi, gi)
        for pi, p in enumerate(pred_boxes)
        for gi, g in enumerate(gt_boxes)
        if _iou(p, g) >= iou_thresh
    )
    mp = set(); mg = set(); pairs = []
    for _, pi, gi in candidates:
        if pi not in mp and gi not in mg:
            pairs.append((pi, gi)); mp.add(pi); mg.add(gi)
    return (pairs,
            [i for i in range(len(pred_boxes)) if i not in mp],
            [i for i in range(len(gt_boxes))   if i not in mg])


# ── Lightweight evaluation with pluggable matcher ─────────────────────────────
def _eval_with_matcher(preds_by_img, gt_dict, classes, matcher):
    """Returns detection P/R/F1 + per-class TP/FP/FN using the given matcher."""
    det_tp = det_fp = det_fn = 0
    tp_c = defaultdict(int); fp_c = defaultdict(int); fn_c = defaultdict(int)
    for img_p, gt_boxes in gt_dict.items():
        insect = [p for p in preds_by_img.get(img_p, []) if not p.get('is_bg')]
        pairs, unp, ung = matcher(insect, gt_boxes)
        det_tp += len(pairs); det_fp += len(unp); det_fn += len(ung)
        for pi, gi in pairs: tp_c[gt_boxes[gi]['cls']] += 1
        for pi in unp:       fp_c[insect[pi]['cls']]  += 1
        for gi in ung:       fn_c[gt_boxes[gi]['cls']] += 1
    p  = det_tp / max(1, det_tp + det_fp)
    rc = det_tp / max(1, det_tp + det_fn)
    f1 = 2 * p * rc / max(1e-8, p + rc)
    return {'det_precision': p, 'det_recall': rc, 'det_f1': f1,
            'det_tp': det_tp, 'det_fp': det_fp, 'det_fn': det_fn,
            'tp_c': dict(tp_c), 'fp_c': dict(fp_c), 'fn_c': dict(fn_c)}


# ── Run both matchers on every pipeline ──────────────────────────────────────
_MATCHERS = {
    'center_match': center_match,
    'IoU>=0.50':    lambda p, g: iou_match(p, g, 0.50),
}

_match_results = {}  # label -> {matcher_name -> metrics}

for _lbl, _pbi in pred_indexes.items():
    if _lbl.endswith('_no_bb'): continue
    _match_results[_lbl] = {
        name: _eval_with_matcher(_pbi, gt, GT_CLASSES, fn)
        for name, fn in _MATCHERS.items()
    }

# ── Overall comparison table ──────────────────────────────────────────────────
print('\n' + '='*100)
print('Matching Method Comparison — center_match vs IoU ≥ 0.5')
print('='*100)

_hdr = f'  {"Pipeline":38}  {"Matcher":14}  {"P":>7}  {"R":>7}  {"F1":>7}  {"TP":>5}  {"FP":>5}  {"FN":>5}'
print(f'\n{_hdr}')
print('  ' + '-'*92)
for _lbl, _methods in _match_results.items():
    for _tag, _r in _methods.items():
        print(f'  {_lbl:38}  {_tag:14}  '
              f'{_r["det_precision"]:>7.3f}  {_r["det_recall"]:>7.3f}  {_r["det_f1"]:>7.3f}  '
              f'{_r["det_tp"]:>5}  {_r["det_fp"]:>5}  {_r["det_fn"]:>5}')
    print()

# ── Per-class recall table ────────────────────────────────────────────────────
print('\nPer-class recall by matcher:')
for _lbl, _methods in _match_results.items():
    print(f'\n  {_lbl}')
    print(f'  {"Class":15}  {"center_match R":>15}  {"IoU>=0.5 R":>11}  {"Delta":>7}')
    print(f'  {"-"*52}')
    for _c in GT_CLASSES:
        _cm = _methods['center_match']
        _io = _methods['IoU>=0.50']
        _cm_tp = _cm['tp_c'].get(_c, 0); _cm_fn = _cm['fn_c'].get(_c, 0)
        _io_tp = _io['tp_c'].get(_c, 0); _io_fn = _io['fn_c'].get(_c, 0)
        _cm_rc = _cm_tp / max(1, _cm_tp + _cm_fn)
        _io_rc = _io_tp / max(1, _io_tp + _io_fn)
        print(f'  {_c:15}  {_cm_rc:>15.3f}  {_io_rc:>11.3f}  {_cm_rc - _io_rc:>+7.3f}')

# ── Summary: delta recall ─────────────────────────────────────────────────────
print('\nOverall Δ recall (center_match − IoU ≥ 0.5):')
for _lbl, _methods in _match_results.items():
    _cm_rc = _methods['center_match']['det_recall']
    _io_rc = _methods['IoU>=0.50']['det_recall']
    print(f'  {_lbl:38}  Δ = {_cm_rc - _io_rc:+.3f}  '
          f'(center_match={_cm_rc:.3f}  IoU≥0.5={_io_rc:.3f})')


##### Cell 18 — Operating Curve (Recall vs False Positives)

Shows how pipeline recall and false-positive count change as the confidence
threshold is swept from strict (0.95) to permissive (0.01).

- **X-axis**: total false positive count across all images.
- **Y-axis**: recall (TP / total GT insects).
- **Per-class solid lines** + **aggregate dashed line** for each pipeline.
- Plotted for both `center_match` and `IoU ≥ 0.5` matchers side-by-side.

Saved to `EVAL_DIR/operating_curve.png`.

In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
_OC_THRESHOLDS = np.unique(np.concatenate([
    np.linspace(0.01, 0.30, 25),   # dense at low end where recall saturates
    np.linspace(0.31, 0.95, 20),
]))  # 45 points total

_OC_MATCHERS = {
    'center_match': center_match,
    'IoU>=0.50':    lambda p, g: iou_match(p, g, 0.50),
}

_CLASS_COLORS = {c: col for c, col in zip(
    GT_CLASSES,
    ['#e41a1c', '#377eb8', '#ff7f00', '#4daf4a', '#984ea3', '#a65628']
)}
_MATCHER_LS = {'center_match': '-', 'IoU>=0.50': '--'}

# ── Sweep function ────────────────────────────────────────────────────────────
def _oc_sweep(preds_by_img, gt_dict, classes, matcher, thresholds):
    """Returns (overall_recall[], overall_fp[], {cls: recall[]}, {cls: fp[]})."""
    n_gt_total = sum(len(v) for v in gt_dict.values())
    n_gt_cls   = {c: sum(1 for v in gt_dict.values() for b in v if b['cls'] == c)
                  for c in classes}

    ov_rec = []; ov_fp  = []
    pc_rec = {c: [] for c in classes}
    pc_fp  = {c: [] for c in classes}

    for thr in thresholds:
        filtered = {
            img_p: [p for p in preds if not p.get('is_bg') and p['conf'] >= thr]
            for img_p, preds in preds_by_img.items()
        }
        r = _eval_with_matcher(filtered, gt_dict, classes, matcher)
        ov_rec.append(r['det_recall'])
        ov_fp.append(r['det_fp'])
        for c in classes:
            tp = r['tp_c'].get(c, 0); fn = r['fn_c'].get(c, 0)
            pc_rec[c].append(tp / max(1, n_gt_cls.get(c, 1)))
            pc_fp[c].append(r['fp_c'].get(c, 0))

    return ov_rec, ov_fp, pc_rec, pc_fp


# ── Plot ──────────────────────────────────────────────────────────────────────
_pipe_labels = [l for l in pred_indexes if not l.endswith('_no_bb')]
_n_pipes = len(_pipe_labels)
_n_match = len(_OC_MATCHERS)

fig, axes = plt.subplots(
    _n_pipes, _n_match,
    figsize=(7 * _n_match, 5 * _n_pipes),  # ~300 DPI → publication-quality
    squeeze=False
)

print('Computing operating curves (may take ~30 s)...')
_oc_curve_data = {}

for _ri, _lbl in enumerate(_pipe_labels):
    _pbi = pred_indexes[_lbl]
    _oc_curve_data[_lbl] = {}

    for _ci, (_mname, _mfn) in enumerate(_OC_MATCHERS.items()):
        ax = axes[_ri][_ci]
        print(f'  {_lbl} / {_mname}...', end='', flush=True)

        _or, _ofp, _pcr, _pcfp = _oc_sweep(_pbi, gt, GT_CLASSES, _mfn, _OC_THRESHOLDS)
        _oc_curve_data[_lbl][_mname] = {
            'thresholds': _OC_THRESHOLDS.tolist(),
            'overall_recall': _or, 'overall_fp': _ofp,
            'per_class_recall': _pcr, 'per_class_fp': _pcfp,
        }
        print(' done')

        # Overall aggregate (dashed black)
        ax.plot(_ofp, _or, 'k--', lw=2.5, label='Overall', zorder=5)

        # Per-class solid lines
        for _c in GT_CLASSES:
            _col = _CLASS_COLORS.get(_c, 'grey')
            ax.plot(_pcfp[_c], _pcr[_c], color=_col, lw=1.8, label=_c)

        # Annotate a few threshold values along the overall curve
        for _ann_thr in [0.10, 0.30, 0.50, 0.70]:
            _idx = int(np.argmin(np.abs(_OC_THRESHOLDS - _ann_thr)))
            ax.annotate(f'{_ann_thr:.2f}',
                        (_ofp[_idx], _or[_idx]),
                        textcoords='offset points', xytext=(4, 2),
                        fontsize=7, color='black', alpha=0.75)

        ax.set_xlabel('False Positives (count)', fontsize=11)
        ax.set_ylabel('Recall', fontsize=11)
        ax.set_title(f'{_lbl}\n{_mname}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=9, loc='lower right', framealpha=0.8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05); ax.set_xlim(left=0)

plt.suptitle(
    'Operating Curve — Recall vs False Positives\n'
    '(annotated numbers = confidence threshold; curve moves right as threshold decreases)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
_oc_path = EVAL_DIR / 'operating_curve.png'
plt.savefig(_oc_path, dpi=300, bbox_inches='tight')
print(f'\n✓ Saved {_oc_path}')
plt.show()


##### Cell 19 — Complementarity Bar Chart (crop vs YOLO)

For every GT insect in the evaluation set, classifies whether it was found by:

- **Both** pipelines
- **Crop only** (missed by YOLO)
- **YOLO only** (missed by crop)
- **Neither** (completely missed)

Broken down per class and as a total.  The *union recall* column shows what a combined pipeline would achieve.

Saved to `EVAL_DIR/complementarity.png`.

In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.transforms import blended_transform_factory

print('\n' + '='*80)
print('Complementarity Analysis — crop pipeline vs YOLO')
print('='*80)

_yolo_labels_comp = list(yolo_run_data.keys())
if not _yolo_labels_comp:
    print('⚠️  No YOLO runs loaded — skipping complementarity chart.')
else:
    _YOLO_LBL_C = _yolo_labels_comp[0]
    _yolo_preds_c = pred_indexes.get(_YOLO_LBL_C, {})
    _crop_labels_c = [l for l in pred_indexes
                      if l != _YOLO_LBL_C and not l.endswith('_no_bb')]

    _KEYS  = ('both', 'crop_only', 'yolo_only', 'neither')

    _comp_store = {}  # crop_label -> {class -> {key -> count}}

    for _cl in _crop_labels_c:
        _cpreds = pred_indexes[_cl]
        _per_cls = {c: {k: 0 for k in _KEYS} for c in GT_CLASSES + ['_total']}

        for _img_p, _gt_boxes in gt.items():
            _ci  = [p for p in _cpreds.get(_img_p, [])       if not p.get('is_bg')]
            _yi  = _yolo_preds_c.get(_img_p, [])
            _c_pairs, _, _c_ung = center_match(_ci, _gt_boxes)
            _y_pairs, _, _y_ung = center_match(_yi, _gt_boxes)
            _c_gi_cls = {_gi2: _ci[_pi2]['cls'] for _pi2,_gi2 in _c_pairs}
            _y_gi_cls = {_gi2: _yi[_pi2]['cls'] for _pi2,_gi2 in _y_pairs}

            for _gi, _gb in enumerate(_gt_boxes):
                _gc = _gb['cls']
                _in_c = _gi in _c_gi_cls and _c_gi_cls[_gi] == _gc
                _in_y = _gi in _y_gi_cls and _y_gi_cls[_gi] == _gc
                _key = ('both' if (_in_c and _in_y) else
                        'crop_only' if _in_c else
                        'yolo_only' if _in_y else 'neither')
                _per_cls[_gc][_key]      += 1
                _per_cls['_total'][_key] += 1

        _comp_store[_cl] = _per_cls

        # ── Print table ───────────────────────────────────────────────────────
        print(f'\n  {_cl}  +  {_YOLO_LBL_C}')
        print(f'  {"Class":12}  {"Both":>6}  {"Crop only":>10}  {"YOLO only":>10}  '
              f'{"Neither":>8}  {"Total":>6}  {"Union R":>8}')
        print(f'  {"-"*70}')
        for _gc in GT_CLASSES + ['_total']:
            _d = _per_cls[_gc]; _tot = sum(_d.values())
            if _tot == 0: continue
            _union_r = (_d['both'] + _d['crop_only'] + _d['yolo_only']) / max(1, _tot)
            _gname = 'TOTAL' if _gc == '_total' else _gc
            print(f'  {_gname:12}  {_d["both"]:>6}  {_d["crop_only"]:>10}  '
                  f'{_d["yolo_only"]:>10}  {_d["neither"]:>8}  {_tot:>6}  {_union_r:>8.3f}')

    # ── GT counts per class (for fraction denominator) ────────────────────────
    _gt_counts = {_gc: 0 for _gc in GT_CLASSES}
    for _boxes in gt.values():
        for _b in _boxes:
            if _b['cls'] in _gt_counts:
                _gt_counts[_b['cls']] += 1
    _gt_total = sum(_gt_counts.values())

    # class display order: GT_CLASSES + All insects
    _cls_display  = GT_CLASSES + ['_total']
    _cls_totals   = [_gt_counts.get(_gc, 1) for _gc in GT_CLASSES] + [_gt_total]
    _cls_xlabels  = [f'{_gc.capitalize()}\n(n={_gt_counts[_gc]})' for _gc in GT_CLASSES] + \
                    [f'All insects\n(n={_gt_total})']

    # ── Thesis-style stacked bar chart ────────────────────────────────────────
    _c_both = '#156215'
    _c_yolo = '#155a8a'
    _c_crop = '#c05a00'
    _c_miss = '#aaaaaa'

    _legend_handles = [
        mpatches.Patch(facecolor=_c_both, label='Detected by both'),
        mpatches.Patch(facecolor=_c_yolo, label='YOLO only'),
        mpatches.Patch(facecolor=_c_crop, label='Crop pipeline only'),
        mpatches.Patch(facecolor=_c_miss, label='Missed by both'),
    ]

    _x = np.arange(len(_cls_display))
    _w = 0.36
    _n_panels = len(_crop_labels_c)

    # figure height: 6.5 per panel, plus 0.5 margin
    _H_ax   = 0.32
    _gap    = 0.22
    _fig_h  = 12.0 if _n_panels <= 2 else 6.5 * _n_panels
    _fig    = plt.figure(figsize=(10.0, _fig_h))

    # compute panel bottom positions (top → bottom)
    _bottoms = []
    for _pi in range(_n_panels):
        _bottoms.append(1.0 - (_pi + 1) * (_H_ax + _gap) + _gap * 0.6)
    # clamp so lowest panel starts at 0.07
    _shift = _bottoms[-1] - 0.07
    _bottoms = [_b - _shift for _b in _bottoms]

    for _pi, _cl in enumerate(_crop_labels_c):
        _pos = [0.10, _bottoms[_pi], 0.87, _H_ax]
        ax = _fig.add_axes(_pos)
        _per = _comp_store[_cl]

        # fractions
        def _frac(_gc, _key):
            _tot = _cls_totals[_cls_display.index(_gc)]
            return _per[_gc][_key] / max(1, _tot)

        _both_f     = [_frac(_gc, 'both')     for _gc in _cls_display]
        _yolo_only_f= [_frac(_gc, 'yolo_only')for _gc in _cls_display]
        _crop_only_f= [_frac(_gc, 'crop_only')for _gc in _cls_display]
        _r_comb     = [(_per[_gc]['both'] + _per[_gc]['crop_only'] + _per[_gc]['yolo_only'])
                       / max(1, _cls_totals[_cls_display.index(_gc)])
                       for _gc in _cls_display]
        _r_crop     = [(_per[_gc]['both'] + _per[_gc]['crop_only'])
                       / max(1, _cls_totals[_cls_display.index(_gc)])
                       for _gc in _cls_display]
        _r_yolo     = [(_per[_gc]['both'] + _per[_gc]['yolo_only'])
                       / max(1, _cls_totals[_cls_display.index(_gc)])
                       for _gc in _cls_display]
        _missed_f   = [1 - _r_comb[_i] for _i in range(len(_cls_display))]

        _bot_y = list(_both_f)
        _bot_c = [_both_f[_i] + _yolo_only_f[_i] for _i in range(len(_cls_display))]

        ax.bar(_x, _both_f,      width=_w, color=_c_both, zorder=3)
        ax.bar(_x, _yolo_only_f, width=_w, color=_c_yolo, bottom=_bot_y,  zorder=3)
        ax.bar(_x, _crop_only_f, width=_w, color=_c_crop, bottom=_bot_c,  zorder=3)
        ax.bar(_x, _missed_f,    width=_w, color=_c_miss, bottom=_r_comb, zorder=3)

        # recall annotations above bars
        _trans = blended_transform_factory(ax.transData, ax.transAxes)
        _ann   = 8.5
        for _i in range(len(_cls_display)):
            ax.text(_x[_i], 1.030, f"crop: {_r_crop[_i]:.3f}",
                    transform=_trans, ha='center', va='bottom',
                    fontsize=_ann-0.5, color=_c_crop)
            ax.text(_x[_i], 1.090, f"YOLO: {_r_yolo[_i]:.3f}",
                    transform=_trans, ha='center', va='bottom',
                    fontsize=_ann-0.5, color=_c_yolo)
            ax.text(_x[_i], 1.150, f"combined: {_r_comb[_i]:.3f}",
                    transform=_trans, ha='center', va='bottom',
                    fontsize=_ann, color='#111', fontweight='bold')

        ax.set_xticks(_x)
        ax.set_xticklabels(_cls_xlabels, fontsize=10.5)
        ax.set_ylabel('Fraction of ground truth insects', fontsize=10)
        ax.set_ylim(0, 1.0)
        ax.set_xlim(-0.55, len(_cls_display) - 0.45)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda _v, _: f'{_v:.1f}'))
        ax.spines[['top','right']].set_visible(False)
        ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)
        ax.tick_params(axis='y', labelsize=9)

        ax.legend(handles=_legend_handles,
                  loc='lower center',
                  bbox_to_anchor=(0.5, 1.210),
                  bbox_transform=ax.transAxes,
                  ncol=4, fontsize=8.5, frameon=True,
                  framealpha=0.92, edgecolor='#ccc',
                  handlelength=1.2, handletextpad=0.5, columnspacing=1.0)

        ax.text(0.5, 1.315, f'{_cl}  +  {_YOLO_LBL_C}',
                transform=ax.transAxes, ha='center', va='bottom',
                fontsize=10.5, fontweight='bold', color='#111', clip_on=False)

    _comp_path = EVAL_DIR / 'complementarity.png'
    _fig.savefig(_comp_path, dpi=200, bbox_inches='tight')
    plt.close(_fig)
    print(f'\n✓ Saved {_comp_path}')


##### Cell 20 — Thesis Operating Curve (all pipelines)

Regenerates `thesis_operating_curve_all.png` inside `EVAL_DIR`.  
Sweeps YOLO confidence thresholds and plots recall vs. false positives for:
* each crop pipeline combined with YOLO
* each crop pipeline alone
* YOLO alone

Uses the same GT, YOLO preds, and crop preds already loaded in earlier cells.


In [ ]:
# ── Thesis Operating Curve ───────────────────────────────────────────────────
# Adapted from figures/make_operating_curve.py
# Uses variables already defined in this notebook:
#   IMAGE_ROOT, GT_ANN_ROOT, YOLO_RUNS, CROP_RUNS, STRIP_HEIGHT, GT_CLASSES, EVAL_DIR

import csv as _oc_csv
import numpy as _oc_np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as _oc_plt
from collections import defaultdict as _oc_dd
from pathlib import Path as _oc_Path

_oc_YOLO_RUN = list(YOLO_RUNS.values())[0]
_oc_CROP_RUN = list(CROP_RUNS.values())[0]
_oc_OUT      = EVAL_DIR / 'thesis_operating_curve_all.png'

# ── GT loading ───────────────────────────────────────────────────────────────
def _oc_load_gt(ann_root, img_root, strip_h=120):
    """Returns {img_path_str: [{'cls','x1','y1','x2','y2'}, ...]}"""
    gt = {}
    for cam_dir in sorted(_oc_Path(ann_root).iterdir()):
        if not cam_dir.is_dir(): continue
        lbl_dir = cam_dir / 'obj_train_data'
        img_dir = _oc_Path(img_root) / cam_dir.name
        if not lbl_dir.exists() or not img_dir.exists(): continue
        names_f = cam_dir / 'obj.names'
        cls_names = ([l.strip() for l in names_f.read_text().splitlines() if l.strip()]
                     if names_f.exists() else GT_CLASSES)
        for txt in sorted(lbl_dir.glob('*.txt')):
            img_p = None
            for ext in ('.JPG', '.jpg', '.jpeg', '.png'):
                cand = img_dir / (txt.stem + ext)
                if cand.exists(): img_p = cand; break
            if img_p is None: continue
            import cv2 as _oc_cv2
            img = _oc_cv2.imread(str(img_p))
            if img is None: continue
            H, W = img.shape[:2]
            boxes = []
            for line in txt.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5: continue
                try:
                    ci = int(parts[0])
                    cx, cy, bw, bh = [float(v) for v in parts[1:5]]
                except ValueError:
                    continue
                cname = cls_names[ci] if ci < len(cls_names) else str(ci)
                if cname not in GT_CLASSES: continue
                x1 = (cx - bw/2)*W;  x2 = (cx + bw/2)*W
                y1 = (cy - bh/2)*H;  y2 = (cy + bh/2)*H
                if y1 > H - strip_h: continue
                boxes.append({'cls': cname, 'x1':x1, 'y1':y1, 'x2':x2, 'y2':y2})
            if boxes:
                gt[str(img_p)] = boxes
    return gt

# ── Center-match helper ──────────────────────────────────────────────────────
def _oc_center_match(preds, gts):
    def _inside(px, py, x1, y1, x2, y2): return x1<=px<=x2 and y1<=py<=y2
    def _ovr(p, g):
        iw = max(0, min(p['x2'],g['x2']) - max(p['x1'],g['x1']))
        ih = max(0, min(p['y2'],g['y2']) - max(p['y1'],g['y1']))
        ga = max(1, (g['x2']-g['x1'])*(g['y2']-g['y1']))
        return iw*ih/ga
    mp=set(); mg=set(); pairs=[]
    for gi, g in enumerate(gts):
        gcx=(g['x1']+g['x2'])/2; gcy=(g['y1']+g['y2'])/2
        for pi, p in enumerate(preds):
            if pi in mp: continue
            pcx=(p['x1']+p['x2'])/2; pcy=(p['y1']+p['y2'])/2
            if (_inside(gcx,gcy,p['x1'],p['y1'],p['x2'],p['y2']) or
                _inside(pcx,pcy,g['x1'],g['y1'],g['x2'],g['y2']) or
                _ovr(p,g) >= 0.20):
                pairs.append((pi,gi)); mp.add(pi); mg.add(gi); break
    return (pairs,
            [i for i in range(len(preds)) if i not in mp],
            [i for i in range(len(gts))  if i not in mg])

# ── YOLO predictions ─────────────────────────────────────────────────────────
def _oc_load_yolo_preds(yolo_run, img_root):
    preds = _oc_dd(list)
    for cam_dir in sorted(_oc_Path(yolo_run).iterdir()):
        if not cam_dir.is_dir(): continue
        csv_f = cam_dir / 'yolo_results.csv'
        if not csv_f.exists(): continue
        img_dir = _oc_Path(img_root) / cam_dir.name
        with open(csv_f) as f:
            for row in _oc_csv.DictReader(f):
                img_name = row['image_name']
                img_p = None
                for ext in ('.JPG','.jpg','.jpeg','.png'):
                    cand = img_dir / (_oc_Path(img_name).stem + ext)
                    if cand.exists(): img_p = cand; break
                if img_p is None:
                    img_p = img_dir / img_name
                x = int(row['bbox_x']); y_top = int(row['bbox_y'])
                w = int(row['bbox_w']); h_ = int(row['bbox_h'])
                preds[str(img_p)].append({
                    'cls': row['class_name'], 'conf': float(row['confidence']),
                    'x1': x, 'y1': y_top, 'x2': x+w, 'y2': y_top+h_,
                })
    return dict(preds)

# ── Crop predictions ─────────────────────────────────────────────────────────
def _oc_load_crop_preds(crop_run, pipe_name, img_root):
    preds = _oc_dd(list)
    p = pipe_name + '__'
    for csv_f in sorted(_oc_Path(crop_run).rglob('results.csv')):
        cam_name = csv_f.parent.name
        img_dir  = _oc_Path(img_root) / cam_name
        with open(csv_f) as f:
            for row in _oc_csv.DictReader(f):
                if row.get('pollinator_detected') != 'yes': continue
                bl = row.get(p + 'binary_label',   '')
                pt = row.get(p + 'pollinator_type', '')
                if (bl == 'background') or (pt == 'background') or (not bl and not pt):
                    continue
                try:
                    conf = float(row.get(p + 'group_conf') or row.get(p + 'binary_conf') or 0)
                except (ValueError, TypeError):
                    conf = 0.0
                cls = pt if (pt and pt != 'background') else bl
                img_name = row.get('image_name', '')
                img_p = None
                for ext in ('.JPG', '.jpg', '.jpeg', '.png'):
                    cand = img_dir / (_oc_Path(img_name).stem + ext)
                    if cand.exists(): img_p = cand; break
                if img_p is None:
                    img_p = img_dir / img_name
                try:
                    x1 = float(row['bbox_x']); y1 = float(row['bbox_y'])
                    x2 = x1 + float(row['bbox_w']); y2 = y1 + float(row['bbox_h'])
                except (ValueError, KeyError):
                    continue
                preds[str(img_p)].append({'cls': cls, 'conf': conf,
                                           'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2})
    return dict(preds)

# ── Fly stats ────────────────────────────────────────────────────────────────
def _oc_fly_stats(preds_by_img, gt_by_img, conf_thr):
    fly_tp = 0; fly_fp = 0
    for img_p, gt_boxes in gt_by_img.items():
        filtered  = [p for p in preds_by_img.get(img_p, []) if p['conf'] >= conf_thr]
        fly_preds = [p for p in filtered if p['cls'] == 'fly']
        gt_fly    = [g for g in gt_boxes  if g['cls'] == 'fly']
        pairs, unm_p, _ = _oc_center_match(fly_preds, gt_fly)
        fly_tp += len(pairs); fly_fp += len(unm_p)
    return fly_tp, fly_fp

def _oc_combined_fly_stats(crop_by_img, yolo_by_img, gt_by_img, yolo_thr):
    fly_tp = 0
    for img_p, gt_boxes in gt_by_img.items():
        crop_fly = [p for p in crop_by_img.get(img_p, []) if p['cls'] == 'fly']
        yolo_fly = [p for p in yolo_by_img.get(img_p, [])
                    if p['cls'] == 'fly' and p['conf'] >= yolo_thr]
        gt_fly   = [g for g in gt_boxes if g['cls'] == 'fly']
        pairs, _, _ = _oc_center_match(crop_fly + yolo_fly, gt_fly)
        fly_tp += len(pairs)
    return fly_tp

# ── Build the figure ─────────────────────────────────────────────────────────
print('Loading GT for operating curve...')
_oc_gt = _oc_load_gt(GT_ANN_ROOT, IMAGE_ROOT, STRIP_HEIGHT)
_oc_fly_gt_n = sum(1 for boxes in _oc_gt.values() for b in boxes if b['cls']=='fly')
print(f'  GT fly instances: {_oc_fly_gt_n}')

print('Loading YOLO predictions...')
_oc_yolo_preds = _oc_load_yolo_preds(_oc_YOLO_RUN, IMAGE_ROOT)

_oc_PIPELINES = {
    'two_stage':      ('Two-step crop pipeline', 'two_stage'),
    'five_class_ins': ('Five-class InsectNet crop pipeline', 'five_class_ins'),
}
_oc_THRESHOLDS = _oc_np.linspace(0.70, 0.05, 9)

C_TS  = '#1f77b4'
C_INS = '#d62728'
C_YLO = '#7f7f7f'

_oc_fig, _oc_ax = _oc_plt.subplots(figsize=(11, 8.0))
_oc_e2e_thr = 0.20
_oc_pipe_data = {}

for _oc_pk, (_oc_plabel, _oc_pcsv) in _oc_PIPELINES.items():
    print(f'Loading crop predictions: {_oc_pcsv}...')
    _oc_cpreds = _oc_load_crop_preds(_oc_CROP_RUN, _oc_pcsv, IMAGE_ROOT)
    _oc_color  = {'two_stage': C_TS, 'five_class_ins': C_INS}[_oc_pk]
    _oc_short  = {'two_stage': 'Two-step', 'five_class_ins': 'Five-class InsectNet'}[_oc_pk]

    _oc_recalls = []; _oc_fps = []
    for _oc_thr in _oc_THRESHOLDS:
        _oc_ytp, _oc_yfp = _oc_fly_stats(_oc_yolo_preds, _oc_gt, _oc_thr)
        _oc_ctp = _oc_combined_fly_stats(_oc_cpreds, _oc_yolo_preds, _oc_gt, _oc_thr)
        _oc_recalls.append(_oc_ctp / _oc_fly_gt_n)
        _oc_fps.append(_oc_yfp)
    _oc_pipe_data[_oc_pk] = (_oc_fps, _oc_recalls)

    _oc_ax.plot(_oc_fps, _oc_recalls, '-o', color=_oc_color, lw=2.2, ms=5,
                label=f'{_oc_short} + YOLO')
    _oc_label_above = (_oc_pk == 'two_stage')
    for _oc_lt in [0.70, 0.20, 0.05]:
        _oc_idx = int(_oc_np.argmin(_oc_np.abs(_oc_THRESHOLDS - _oc_lt)))
        _oc_ax.plot(_oc_fps[_oc_idx], _oc_recalls[_oc_idx], 'o', color=_oc_color, ms=9, zorder=4)
        _oc_dy = 0.013 if _oc_label_above else -0.022
        _oc_ax.text(_oc_fps[_oc_idx], _oc_recalls[_oc_idx] + _oc_dy, f'{_oc_lt:.2f}',
                    ha='center', va='bottom' if _oc_label_above else 'top',
                    fontsize=15, color=_oc_color)

    # Crop-alone sweep
    _oc_cr_sweep = []; _oc_cf_sweep = []
    for _oc_thr in _oc_THRESHOLDS:
        _oc_ctp, _oc_cfp = _oc_fly_stats(_oc_cpreds, _oc_gt, _oc_thr)
        _oc_cr_sweep.append(_oc_ctp / _oc_fly_gt_n)
        _oc_cf_sweep.append(_oc_cfp)
    _oc_ax.plot(_oc_cf_sweep, _oc_cr_sweep, '--^', color=_oc_color,
                lw=1.4, ms=5, alpha=0.8, label=f'{_oc_short} alone')
    for _oc_lt, _oc_above in ([(0.70,True),(0.20,True)] if _oc_pk=='two_stage'
                               else [(0.20,True),(0.70,False)]):
        _oc_idx = int(_oc_np.argmin(_oc_np.abs(_oc_THRESHOLDS - _oc_lt)))
        _oc_dy2 = 0.015 if _oc_above else -0.020
        _oc_ax.plot(_oc_cf_sweep[_oc_idx], _oc_cr_sweep[_oc_idx], '^',
                    color=_oc_color, ms=10, zorder=5, alpha=0.9)
        _oc_ax.text(_oc_cf_sweep[_oc_idx], _oc_cr_sweep[_oc_idx] + _oc_dy2,
                    f'{_oc_lt:.2f}', ha='center',
                    va='bottom' if _oc_above else 'top',
                    fontsize=15, color=_oc_color, alpha=0.9)

# YOLO alone
_oc_yr = []; _oc_yf = []
for _oc_thr in _oc_THRESHOLDS:
    _oc_ytp, _oc_yfp = _oc_fly_stats(_oc_yolo_preds, _oc_gt, _oc_thr)
    _oc_yr.append(_oc_ytp / _oc_fly_gt_n); _oc_yf.append(_oc_yfp)
_oc_ax.plot(_oc_yf, _oc_yr, '--s', color=C_YLO, ms=5, lw=1.6, label='YOLO only', zorder=2)
for _oc_lt in [0.70, 0.20, 0.05]:
    _oc_idx = int(_oc_np.argmin(_oc_np.abs(_oc_THRESHOLDS - _oc_lt)))
    _oc_ax.plot(_oc_yf[_oc_idx], _oc_yr[_oc_idx], 's', color=C_YLO, ms=7, zorder=3)
    _oc_dy3 = 0.012 if _oc_lt == 0.30 else -0.022
    _oc_ax.text(_oc_yf[_oc_idx], _oc_yr[_oc_idx] + _oc_dy3, f'{_oc_lt:.2f}',
                ha='center', va='bottom' if _oc_dy3 > 0 else 'top',
                fontsize=15, color='#555')

# Operating-point marker
_oc_ts_fps, _oc_ts_rec = _oc_pipe_data['two_stage']
_oc_idx = int(_oc_np.argmin(_oc_np.abs(_oc_THRESHOLDS - _oc_e2e_thr)))
_oc_ex = _oc_ts_fps[_oc_idx]; _oc_ey = _oc_ts_rec[_oc_idx]
_oc_ax.plot(_oc_ex, _oc_ey, 'o', ms=14, mfc='none', mec='#111', mew=2.0, zorder=5)
_oc_ax.annotate(
    f'Web app deployed\n(YOLO conf = {_oc_e2e_thr:.2f})',
    xy=(_oc_ex, _oc_ey), xytext=(_oc_ex+240, _oc_ey-0.14),
    fontsize=15, color='#111', ha='center',
    arrowprops=dict(arrowstyle='->', color='#111', lw=1.4))

_oc_ax.set_xlabel('Number of false positives to review', fontsize=21, labelpad=14)
_oc_ax.set_ylabel(f'Fly recall  (GT n = {_oc_fly_gt_n})', fontsize=21, labelpad=14)
_oc_ax.set_title(
    'Fly recall vs. review burden\nas YOLO confidence threshold varies',
    fontsize=22, fontweight='bold', pad=18)
_oc_ax.legend(fontsize=18, framealpha=0.9, loc='lower right')
_oc_ax.tick_params(axis='both', labelsize=14)
_oc_ax.grid(alpha=0.25)
_oc_ax.spines[['top', 'right']].set_visible(False)
_oc_ax.set_ylim(bottom=0.22, top=0.82)

_oc_fig.tight_layout()
_oc_OUT.parent.mkdir(parents=True, exist_ok=True)
_oc_fig.savefig(_oc_OUT, dpi=200, bbox_inches='tight')
_oc_plt.close(_oc_fig)
print(f'\n✓  Saved {_oc_OUT}')


##### Cell 21 — FN Crop Export

Exports image crops of every false-negative (missed) GT insect for qualitative inspection.

Organisation: `EVAL_DIR/fn_crops/<pipeline>/<fn_type>/<class>/<img_stem>_NNN.jpg`

- `fn_not_detected`: no predicted bbox came near this GT insect.
- `fn_detected_as_bg`: a bbox overlapped but the crop was classified as background.

Adjust `_FN_MARGIN` to control how much context is included around each GT box.

In [ ]:
import cv2
from pathlib import Path

_FN_MARGIN  = 20   # extra pixels of context around each GT box
_FN_CROP_DIR = EVAL_DIR / 'fn_crops'
_fn_counts   = {}  # label -> {class -> count}

print('\n' + '='*70)
print('FN Crop Export')
print('='*70)

for _lbl, _pbi in pred_indexes.items():
    if _lbl.endswith('_no_bb'): continue

    _out_root = _FN_CROP_DIR / _lbl.replace('/', '_')
    for _fn_t in ('fn_not_detected', 'fn_detected_as_bg'):
        for _c in GT_CLASSES:
            (_out_root / _fn_t / _c).mkdir(parents=True, exist_ok=True)

    _cnt = {c: 0 for c in GT_CLASSES}

    for _img_p, _gt_boxes in gt.items():
        if not _gt_boxes: continue

        _insect   = [p for p in _pbi.get(_img_p, []) if not p.get('is_bg')]
        _rejected = [p for p in _pbi.get(_img_p, []) if     p.get('is_bg')]
        _pairs, _, _ung = center_match(_insect, _gt_boxes)

        if not _ung: continue

        _img = cv2.imread(str(_img_p))
        if _img is None: continue
        _H, _W = _img.shape[:2]

        for _gi in _ung:
            _gb  = _gt_boxes[_gi]
            _gc  = _gb['cls']
            # Decide FN type: was a rejected bbox nearby?
            _covered = any(
                bbox_overlap_ratio(_r, _gb) >= 0.20 or
                bbox_overlap_ratio(_gb, _r) >= 0.20
                for _r in _rejected
            )
            _fn_type = 'fn_detected_as_bg' if _covered else 'fn_not_detected'

            # Crop with margin
            _x1 = max(0,   int(_gb['x1']) - _FN_MARGIN)
            _y1 = max(0,   int(_gb['y1']) - _FN_MARGIN)
            _x2 = min(_W,  int(_gb['x2']) + _FN_MARGIN)
            _y2 = min(_H,  int(_gb['y2']) + _FN_MARGIN)
            _crop = _img[_y1:_y2, _x1:_x2]
            if _crop.size == 0: continue

            _n  = _cnt.get(_gc, 0)
            _stem = Path(_img_p).stem
            _out_p = _out_root / _fn_type / _gc / f'{_stem}_{_n:04d}.jpg'
            cv2.imwrite(str(_out_p), _crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
            _cnt[_gc] = _n + 1

    _fn_counts[_lbl] = dict(_cnt)

# ── Summary ───────────────────────────────────────────────────────────────────
print('\nExported FN crops:')
for _lbl, _cc in _fn_counts.items():
    _total = sum(_cc.values())
    print(f'\n  {_lbl}  ({_total} total)')
    for _c, _n in sorted(_cc.items()):
        print(f'    {_c:15}: {_n} crops')

print(f'\n✓ Crops saved to: {_FN_CROP_DIR}')


##### Cell 22 — Qualitative Failure Panel

Visualises every pipeline's failures by drawing colour-coded bboxes on the original images:

| Colour | Meaning |
|--------|---------|
| **Green** | True positive (GT insect correctly detected) |
| **Red** | False negative — not detected at all |
| **Orange** | False negative — bbox present but classified as background |
| **Blue** | False positive (spurious detection) |

Images are sorted by failure count (most failures first); only the top `_FP_MAX_PANELS` are shown.  Saved to `EVAL_DIR/failure_panel_<pipeline>.png`.

In [ ]:
import cv2
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

_FP_MAX_PANELS = 16   # max images per pipeline panel grid
_FP_THUMB_W    = 600  # resize width for each thumbnail

# BGR colours for OpenCV drawing
_FP_BGR = {
    'tp':               (50,  200,  50),   # green
    'fn_not_detected':  (30,   30, 200),   # red
    'fn_detected_as_bg': (0,  140, 255),   # orange
    'fp':               (220,  60,  60),   # blue
}

def _draw_box(img, x1, y1, x2, y2, color, label, thickness=2):
    """Draw a labelled rectangle on img (in-place)."""
    x1,y1,x2,y2 = int(x1),int(y1),int(x2),int(y2)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
    cv2.rectangle(img, (x1, y1-th-5), (x1+tw+3, y1), color, -1)
    cv2.putText(img, label, (x1+2, y1-3),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1, cv2.LINE_AA)

print('\n' + '='*70)
print('Qualitative Failure Panels')
print('='*70)

for _lbl, _pbi in pred_indexes.items():
    if _lbl.endswith('_no_bb'): continue

    # Collect images with failures, sorted by failure count
    _failure_imgs = []
    for _img_p, _gt_boxes in gt.items():
        _insect   = [p for p in _pbi.get(_img_p, []) if not p.get('is_bg')]
        _rejected = [p for p in _pbi.get(_img_p, []) if     p.get('is_bg')]
        _pairs, _unp, _ung = center_match(_insect, _gt_boxes)
        _n_fail = len(_unp) + len(_ung)
        if _n_fail > 0:
            _failure_imgs.append((_n_fail, _img_p, _gt_boxes,
                                  _insect, _rejected, _pairs, _unp, _ung))
    _failure_imgs.sort(key=lambda x: -x[0])
    _failure_imgs = _failure_imgs[:_FP_MAX_PANELS]

    if not _failure_imgs:
        print(f'  {_lbl}: no failures — skipping.')
        continue
    print(f'  {_lbl}: rendering {len(_failure_imgs)} failure images...')

    _thumbs = []
    for (_, _img_p, _gt_boxes, _insect, _rejected,
         _pairs, _unp, _ung) in _failure_imgs:

        _img = cv2.imread(str(_img_p))
        if _img is None: continue
        _H, _W = _img.shape[:2]

        _matched_gt   = {gi for _, gi in _pairs}
        _matched_pred = {pi for pi, _ in _pairs}

        # Draw GT boxes
        for _gi, _gb in enumerate(_gt_boxes):
            if _gi in _matched_gt:
                _ftype = 'tp'
            else:
                _covered = any(
                    bbox_overlap_ratio(_r, _gb) >= 0.20 or
                    bbox_overlap_ratio(_gb, _r) >= 0.20
                    for _r in _rejected
                )
                _ftype = 'fn_detected_as_bg' if _covered else 'fn_not_detected'
            _draw_box(_img, _gb['x1'], _gb['y1'], _gb['x2'], _gb['y2'],
                      _FP_BGR[_ftype], f"GT:{_gb['cls']}", thickness=2)

        # Draw FP predictions
        for _pi in _unp:
            _pp = _insect[_pi]
            _draw_box(_img, _pp['x1'], _pp['y1'], _pp['x2'], _pp['y2'],
                      _FP_BGR['fp'], f"FP:{_pp['cls']}({_pp['conf']:.2f})", thickness=1)

        # Resize to thumbnail
        _scale = _FP_THUMB_W / _W
        _thumb = cv2.resize(_img, (_FP_THUMB_W, int(_H * _scale)))
        _thumbs.append((Path(_img_p).name, _thumb))

    if not _thumbs: continue

    # Build matplotlib grid
    _n_cols = min(4, len(_thumbs))
    _n_rows = (len(_thumbs) + _n_cols - 1) // _n_cols

    fig, axes = plt.subplots(_n_rows, _n_cols,
                             figsize=(_n_cols * 5, _n_rows * 4))
    _axarr = np.array(axes).reshape(_n_rows, _n_cols)

    for _i, (_fname, _thumb) in enumerate(_thumbs):
        _ax = _axarr[_i // _n_cols][_i % _n_cols]
        _ax.imshow(cv2.cvtColor(_thumb, cv2.COLOR_BGR2RGB))
        _ax.set_title(_fname, fontsize=8, wrap=True)
        _ax.axis('off')

    for _i in range(len(_thumbs), _n_rows * _n_cols):
        _axarr[_i // _n_cols][_i % _n_cols].axis('off')

    _legend_els = [
        Patch(color='#32c832', label='TP — correctly detected'),
        Patch(color='#c81e1e', label='FN — not detected at all'),
        Patch(color='#ff8c00', label='FN — detected but classified as background'),
        Patch(color='#3c3cdc', label='FP — spurious detection'),
    ]
    fig.legend(handles=_legend_els, loc='lower center',
               ncol=4, fontsize=10, bbox_to_anchor=(0.5, 0), framealpha=0.9)

    _title = (f'Failure Panel — {_lbl}\n'
              f'(sorted by failure count; max {_FP_MAX_PANELS} images)')
    plt.suptitle(_title, fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0.08, 1, 1])

    _panel_path = EVAL_DIR / f'failure_panel_{_lbl.replace("/","_")}.png'
    plt.savefig(_panel_path, dpi=150, bbox_inches='tight')
    print(f'    ✓ Saved {_panel_path}')
    plt.show()

print('\n✓ Failure panels complete.')
